# Bruise segmentation — SAVED MODELS: analysis & visualisation

The same figure suite as `bruise_colab_final_analysis.ipynb`, but for the **5 saved
models** in the project's local dirs (loaded via `pipeline`, exactly as
`bruise_colab_inference_demo.ipynb`). **No training** — checkpoints are loaded and scored.

Front end (below) loads the models, sweeps thresholds on val, and scores test (SegFormer +
YOLO custom /255 + YOLO native argmax). Then the analysis sections build every chart:
accuracy distributions, safety/miss, paired subject-bootstrap contrasts, per-subject
heatmap, fairness (ITA), bruise size + the size-conditioned regression, the annotation
ceiling + "model beats Paul", speed, and qualitative galleries (easy→hard, by skin tone,
by bruise size). Colourblind palette; every chart prints its table.

## 1. Mount Google Drive

> ⚠️ **This notebook needs the rebuilt package** (~1.2 GB), which adds the 134 val
> images the sweep fits on. The old ~868 MB zip shipped **test images only** and
> aliased `train_manifest` to the test manifest — a sweep run against it would have
> silently fitted the threshold on the test set. Rebuild with
> `python scripts/32_build_colab_gpu_package.py` and re-upload before running this.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload the zip to {DRIVE_DIR}/ first.'
size_mb = os.path.getsize(ZIP_PATH) / 1e6
print('Found package:', ZIP_PATH, f'({size_mb:.0f} MB)')
assert size_mb > 1000, (
    f'This zip is {size_mb:.0f} MB — that looks like the OLD test-only package. The val '
    'split (needed to fit thresholds without touching test) makes it ~1.2 GB. '
    'Rebuild with scripts/32_build_colab_gpu_package.py and re-upload.')

## 2. Check we actually have a GPU

Timing numbers are meaningless without one. This stops the notebook immediately rather
than letting you collect nonsense. **Runtime → Change runtime type → A100 GPU.**

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU, then re-run.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Copy the zip to local disk and unzip

We copy off Drive first — reading images over the Drive mount adds unpredictable network
latency to preprocessing.

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/pkg.zip')
print(f'Copied in {time.time()-t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time()-t0:.0f}s')

## 4. Install the libraries

Colab already ships torch + CUDA; reinstalling torch would break the GPU build.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Imports and configuration

> ### 🐛 The `cv2` / `ultralytics` mask bug — and why import order does *not* fix it
>
> Older versions of this notebook imported `cv2` before `ultralytics` "on purpose",
> claiming that avoided `cv2.imread(..., IMREAD_GRAYSCALE)` returning `(H,W,1)` instead
> of `(H,W)`. **That workaround is a myth.** Measured on the real masks:
>
> | | `imread` GRAYSCALE result |
> |---|---|
> | cv2 alone, no ultralytics | `(4022, 6024)` ✅ |
> | ultralytics imported **first** | `(4022, 6024, 1)` ❌ |
> | cv2 imported **first** | `(4022, 6024, 1)` ❌ |
>
> Ultralytics monkey-patches `cv2.imread` at import time. Once it is imported *anywhere*
> in the process, the patch is live — order is irrelevant.
>
> **Why this matters more than it looks.** That trailing axis survives albumentations and
> `ToTensorV2`, so `y` arrives as `[B,1,640,640,1]`. Nothing crashes. Instead
> `dice_np(pred[640,640], gt[640,640,1])` **broadcasts** to `[640,640,640]` and returns
> nonsense — a pixel-perfect prediction scores **Dice 63.9 instead of 1.0**.
>
> Fixed at the source: `pipeline/data.py` now squeezes the trailing axis after `imread`
> (a no-op when cv2 is unpatched). The assert in §6 verifies the fix held — it is the
> difference between a real number and a fake one, so it is checked, not assumed.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import copy, sys, json, time
from pathlib import Path
from torch.utils.data import DataLoader

sys.path.insert(0, LOCAL_DIR)
from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.metrics import compute_image_row

paths = load_yaml('configs/paths.yaml')
cfg   = load_yaml('configs/common_train.yaml')
IMG_H = IMG_W = 640                      # every model runs and is scored at 640x640
assert (cfg['img_h'], cfg['img_w']) == (IMG_H, IMG_W)

torch.backends.cudnn.benchmark = True    # autotune conv kernels; input shape is fixed
print('Config loaded. Grid:', IMG_H, 'x', IMG_W)

# Import ultralytics HERE, up front, rather than next to the YOLO loading code. Not for
# ordering reasons -- the header above shows order is irrelevant -- but so that every
# dataloader in this notebook runs under the SAME cv2 patch state. Importing it later
# (next to the model loading) meant the test set was read with an unpatched cv2 and the
# val set with a patched one, so the two GT tensors came out different shapes depending
# on which cells you had run. Same state everywhere is the actual fix; data.py's squeeze
# then makes that state harmless.
import ultralytics
from ultralytics import YOLO
print('ultralytics', ultralytics.__version__, '| cv2', cv2.__version__)

## 6. Load val + test — and prove they don't overlap

`BruiseDataset` is the same loader used in training, and here it is the **only** thing
that reads pixels. Per image it reads the native photo (4022×6024) and its mask,
cv2-resizes both to 640×640 (image bilinear, mask nearest — nearest keeps the mask
strictly 0/1), ImageNet-normalises the image, and returns tensors.

**Every model gets this exact tensor and this exact geometry.** The only later difference
is that `model_input()` (§9) rescales the pixels for YOLO — no second read, no second
resize.

> ### 🛑 The leak guard below is not decoration
> The whole reason the val split is shipped is so the threshold can be fitted somewhere
> other than the set it's scored on. The **previous** package aliased `train_manifest`
> to `fixed_test_manifest` — had we swept against that, the threshold would have been
> fitted on test and then used to report test scores. That wouldn't crash; it would just
> quietly hand back inflated numbers, which is far worse. The asserts check the
> manifests are different files **and** that no image *and no subject* is in both.

> ### 🛑 So is the mask-shape guard
> With ultralytics imported, `cv2.imread` is patched and GT can silently arrive as
> `(640,640,1)`. That doesn't crash either — it broadcasts, and a *pixel-perfect*
> prediction scores **Dice 63.9**. Both splits are checked.

In [ ]:
val_df  = load_fixed_test(paths['val_manifest'])          # 134 images — thresholds fitted here
test_df = load_fixed_test(paths['fixed_test_manifest'])   # 185 images — scored here, never fitted

# --- leak guard -----------------------------------------------------------
assert Path(paths['val_manifest']).resolve() != Path(paths['fixed_test_manifest']).resolve(), (
    'val_manifest and fixed_test_manifest are the SAME FILE. Sweeping would fit the '
    'threshold on the test set. Rebuild the package with scripts/32_build_colab_gpu_package.py.')
img_overlap = set(val_df.image_path) & set(test_df.image_path)
sub_overlap = set(val_df.subject)    & set(test_df.subject)
assert not img_overlap, f'{len(img_overlap)} image(s) in BOTH val and test: {sorted(img_overlap)[:3]}'
assert not sub_overlap, f'{len(sub_overlap)} subject(s) in BOTH val and test: {sorted(sub_overlap)[:3]}'
print('Leak guard passed: 0 shared images, 0 shared subjects.')

val_ds  = BruiseDataset(val_df,  IMG_H, IMG_W, training=False)
test_ds = BruiseDataset(test_df, IMG_H, IMG_W, training=False)

# --- mask-shape guard (see the cv2/ultralytics note in §5) ----------------
# Checked on BOTH datasets, with ultralytics already imported, because a masked GT of
# shape (640,640,1) does not crash -- it broadcasts and silently returns Dice ~63.
for label, ds in (('val', val_ds), ('test', test_ds)):
    _, _y, *_ = ds[0]
    assert _y.shape == (1, IMG_H, IMG_W), (
        f'{label} GT is {tuple(_y.shape)}, expected (1, {IMG_H}, {IMG_W}). The cv2 '
        'trailing-axis patch is live and pipeline/data.py did not squeeze it — every '
        'Dice below would be garbage. Do not trust any number from this run.')
print(f'Mask-shape guard passed: GT is (1, {IMG_H}, {IMG_W}) for both splits.\n')
# --------------------------------------------------------------------------

val_loader  = DataLoader(val_ds,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

x0, y0, *_ = test_ds[0]
print(f'Val  (fit thresholds) : {len(val_df):3d} images, {val_df.subject.nunique()} subjects')
print(f'Test (report scores)  : {len(test_df):3d} images, {test_df.subject.nunique()} subjects')
print(f'Image tensor          : {tuple(x0.shape)}  (cv2-resized 640, ImageNet-normalised)')
print(f'GT mask tensor        : {tuple(y0.shape)}  (0/1)')

## 7. Stage the test images on the GPU, once

The benchmark measures **640 in → 640 out on the GPU**, so the host→device copy must sit
outside the timer. We run the test loader once here and park all 185 preprocessed
tensors in GPU memory (≈0.9 GB), so the timed loop touches nothing but the model.

Val is *not* staged — the sweep isn't timed, so it streams instead and keeps the memory.

In [ ]:
_imgs, _gts, STEMS = [], [], []
for x, y, stems, *_ in test_loader:
    _imgs.append(x.to(DEVICE, non_blocking=True))
    _gts.append(y)                                    # GT stays on CPU; metrics are numpy
    STEMS.extend(stems)

X_TEST = torch.cat(_imgs, dim=0)
GT_640 = torch.cat(_gts, dim=0)
del _imgs, _gts

assert X_TEST.shape == (len(test_df), 3, IMG_H, IMG_W), X_TEST.shape
assert GT_640.shape == (len(test_df), 1, IMG_H, IMG_W), GT_640.shape
assert len(STEMS) == len(test_df)
print(f'Staged on GPU : {tuple(X_TEST.shape)}  ({X_TEST.element_size()*X_TEST.nelement()/1e9:.2f} GB)')
print(f'GT on CPU     : {tuple(GT_640.shape)}')
print(f'Stems         : {len(STEMS)}  (e.g. {STEMS[0]})')

## 8. Load the 5 trained models — all as raw PyTorch `nn.Module`s

Every model becomes a plain `nn.Module` you call as `model(x)`. No wrapper with hidden
preprocessing, no framework-specific inference API.

**No thresholds are read here.** Each model's `threshold` stays `None` until §10 sweeps
it on val. We *do* read the old `threshold_search.csv` into `csv_threshold`, purely so
§10 can print our sweep next to the historical value as a cross-check.

> ### ⚠️ Licensing — the honest version
> Skipping `.predict()` keeps Ultralytics' **inference/postprocessing** stack out of our
> runtime path. But `best.pt` is a *pickled Ultralytics model object*: deserialising it
> still runs `import ultralytics` and still constructs their classes. **This reduces
> AGPL exposure; it does not remove it.** Only a TorchScript/ONNX export drops the
> import — and even that removes the *technical* dependency, not necessarily the
> *legal* one (whether a graph traced from AGPL source is a derivative work is
> contested, and Ultralytics' own position is that models trained with their code are
> AGPL). Not legal advice.

In [ ]:
import contextlib, warnings
from transformers.utils import logging as hf_logging

@contextlib.contextmanager
def quiet_pretrained_load():
    """Silence HF's load report while building a SegFormer.

    build_segformer() starts from an ImageNet *classification* checkpoint, which has no
    segmentation decoder -- so HF correctly reports every decode_head.* param as MISSING
    and the 1000-class classifier.* as UNEXPECTED. load_segformer_model() then loads our
    fine-tuned state_dict over it with strict key matching, so those params are all
    overwritten before use. The report describes an intermediate state, not the loaded
    model. Errors still raise; only warnings are hidden.
    """
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            yield
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()


def historical_threshold(run_dir):
    """The old threshold_search.csv's best row, as an equivalent T=1 threshold.

    Reference only -- nothing downstream depends on it. The old YOLO grids swept
    (T, thr); since the mask depends only on c = T*logit(thr), the comparable T=1
    threshold is sigmoid(c). For SegFormer (no temperature column) that is just thr.
    Returns None if the file isn't there.
    """
    csv = Path(run_dir) / 'threshold_search.csv'
    if not csv.exists():
        return None
    row = pd.read_csv(csv).sort_values('mean_dice', ascending=False).iloc[0]
    thr = float(row['threshold'])
    temp = float(row['temperature']) if 'temperature' in row.index else 1.0
    cut = temp * np.log(thr / (1.0 - thr))
    return float(1.0 / (1.0 + np.exp(-cut)))


MODELS = {}

# --- SegFormer: 1-channel head ---
for run, disp, pkey in [
    ('segformer_b2_teacher',   'SegFormer-B2 (teacher)',   'segformer_b2_pretrained'),
    ('segformer_b0_direct',    'SegFormer-B0 (direct)',    'segformer_b0_pretrained'),
    ('segformer_b0_distilled', 'SegFormer-B0 (distilled)', 'segformer_b0_pretrained'),
]:
    with quiet_pretrained_load():
        model, _csv_thr, ckpt = load_segformer_model(run, pkey, paths, DEVICE)
    MODELS[run] = {'display': disp, 'model': model.to(DEVICE).eval(),
                   'input': 'imagenet',                     # MiT encoder pretrained ImageNet-normalised
                   'threshold': None,                       # set by the val sweep in §10
                   'csv_threshold': historical_threshold(f'{LOCAL_DIR}/{run}'),
                   'params_m': count_params(model)[0] / 1e6, 'ckpt': str(ckpt)}

# --- YOLO: 2-channel head. Pull the nn.Module out of the wrapper, drop the wrapper.
# YOLO() is used ONLY to unpickle; the wrapper (and its .predict()) is discarded here.
for run, disp in [('yolo_sem_direct',    'YOLO26n-sem (direct)'),
                  ('yolo_sem_distilled', 'YOLO26n-sem (distilled)')]:
    ckpt = f'{LOCAL_DIR}/{run}/ultralytics_runs/train/weights/best.pt'
    nn_model = copy.deepcopy(YOLO(ckpt).model).to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'model': nn_model,
                   'input': '01',                           # Ultralytics trains on /255, NOT ImageNet norm
                   'threshold': None,                       # set by the val sweep in §10
                   'csv_threshold': historical_threshold(f'{LOCAL_DIR}/{run}'),
                   'params_m': sum(p.numel() for p in nn_model.parameters()) / 1e6,
                   'ckpt': ckpt}

for v in MODELS.values():
    hist = 'n/a' if v['csv_threshold'] is None else f"{v['csv_threshold']:.4f}"
    print(f"{v['display']:26s} {v['params_m']:6.2f}M | threshold: pending sweep "
          f"(historical CSV equiv: {hist})")

## 9. The one inference path

Every number in this notebook comes out of these functions. There is no second path.

### Two per-model properties, both structural

`model_input()` branches on **the pixel scale the weights were trained on**, and
`bruise_logit_640()` branches on **the head's channel count**. Neither branches on a
model's name — both are facts about the checkpoint, in the same category as its weights.

| model | trained pixel scale | raw head output | → bruise logit |
|---|---|---|---|
| SegFormer B2/B0 | ImageNet norm | `[1, 1, 160, 160]` (stride 4) | `out[:, 0]` |
| YOLO26n-sem | `/255` | `[1, 2, 80, 80]` (stride 8) | `out[:,1] − out[:,0]` |

The de-normalisation for YOLO is `x*STD + MEAN` — algebraically the exact inverse of what
the dataloader applied, measured accurate to **6e-8**. It changes pixel *scale* only;
resize, interpolation and geometry are untouched.

### About `F.interpolate`

It can't be removed, and it isn't a preprocessing choice — it's structural. **Neither
head emits at 640.** SegFormer's decode head emits 160×160 and `SegformerWrapper`
bilinear-upsamples to 640 *inside its own `forward()`* — that's how it was trained and
scored. YOLO's emits 80×80. The guard applies **the same bilinear upsample to both**: a
no-op for SegFormer, an 8× upsample for YOLO. One operation, both families. The sanity
check prints each head's real shape, so these are observed, not assumed.

What *is* gone: no `cv2.resize` on any model output, no nearest-vs-bilinear mismatch, no
mask ever resized in order to be compared against GT.

### The mask never leaves the GPU

`bruise_mask_640` returns a **CUDA** tensor. The only `.cpu()` is in the accuracy loop,
where numpy metrics genuinely need it — and that loop is not timed.

In [ ]:
# Inverse of the dataloader's ImageNet normalisation. Applied only to models whose
# 'input' is '01' -- see the markdown above for the measurement that forced this.
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def model_input(entry, x):
    """The dataloader tensor, put back on the pixel scale this checkpoint was trained on.

    'imagenet' (SegFormer): as-is -- its MiT encoder was pretrained ImageNet-normalised.
    '01'       (YOLO):      undo the normalisation, recovering the exact /255 image
                            Ultralytics trained on (verified exact to ~6e-8).

    Geometry is untouched either way: same dataloader, same cv2 stretch-resize to 640.
    Only the per-channel scale differs, and that is a property of the weights, not a
    pipeline choice -- feeding YOLO ImageNet-normalised pixels caps it at 0.479 Dice.
    """
    return x * IMAGENET_STD + IMAGENET_MEAN if entry['input'] == '01' else x


def bruise_logit_640(entry, x):
    """[B,3,640,640] normalised GPU tensor -> [B,640,640] bruise logit on the 640 grid.

    Branches on channel count -- a structural fact -- never on a model's name:
      1 channel  (SegFormer): head already emits a bruise logit -> out[:, 0]
      2 channels (YOLO):      2-class softmax == sigmoid(z1-z0) -> out[:,1] - out[:,0]
    """
    out = entry['model'](model_input(entry, x))
    if isinstance(out, (tuple, list)):        # YOLO returns (preds, aux) in eval mode
        out = out[0]
    out = out.float()

    # Same bilinear upsample for both families. No-op for SegFormer (its wrapper already
    # upsampled 160 -> 640 internally); an 8x upsample from 80 -> 640 for YOLO.
    if out.shape[-2:] != (IMG_H, IMG_W):
        out = F.interpolate(out, size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)

    return out[:, 1] - out[:, 0] if out.shape[1] >= 2 else out[:, 0]


def bruise_mask_640(entry, x):
    """[B,3,640,640] GPU tensor -> [B,640,640] bool mask. Stays on the GPU.

    sigmoid(z) >= thr is the same rule as the cut z >= logit(thr) the sweep works in,
    written in probability space because that is the interpretable form.
    """
    if entry['threshold'] is None:
        raise RuntimeError(f"{entry['display']} has no threshold yet — run the val sweep (§10).")
    return torch.sigmoid(bruise_logit_640(entry, x)) >= entry['threshold']


# Sanity check. pred_frac@cut0 is the real guard here: a model fed the wrong pixel scale
# under-fires badly, and that shows up as a predicted-positive fraction far below the GT
# rate -- exactly how the ImageNet-vs-/255 bug was caught (YOLO predicted 0.0118 against
# a GT rate of 0.0473). Numbers far below GT_POS_FRAC mean something is wrong.
GT_POS_FRAC = (GT_640 > 0.5).float().mean().item()
print(f"GT positive fraction: {GT_POS_FRAC:.4f}   (a sane model predicts roughly this)\n")

with torch.inference_mode():
    for m in MODELS.values():
        raw = m['model'](model_input(m, X_TEST[:1]))
        raw = raw[0] if isinstance(raw, (tuple, list)) else raw
        z = bruise_logit_640(m, X_TEST[:8])
        assert z.shape == (8, IMG_H, IMG_W) and z.is_cuda, (z.shape, z.device)
        print(f"{m['display']:26s} in={m['input']:8s} head {str(tuple(raw.shape)):18s} "
              f"-> logit {tuple(z.shape)} | pred_frac@cut0 {(z >= 0).float().mean():.4f}")

---
# 10. Threshold sweep — 1-D, on val, same mechanism for all 5 models

### What is swept

The cut `c` on the raw bruise logit: `mask = (z >= c)`. That is the space the decision
actually lives in. The equivalent probability threshold is `σ(c)`, which is what §11–12
then use — the two are the same rule written two ways.

One forward pass per val image, **cached**, then all cuts scored against the cached
logits. The old sweep re-ran the model for every grid point — 152 × 134 = **20,368**
forward passes per model. This does **134**, then the sweep is pure tensor math.

### How the winner is chosen — *not* `argmax`

The old sweep took `iloc[0]` after sorting by mean_dice. On 134 images its top-8 grid
points spanned **7×10⁻⁴** in mean Dice, with rows 1 and 2 differing by **1.1×10⁻⁵** —
so `iloc[0]` was selecting on noise, not signal.

Instead: compute the standard error of the mean Dice at the peak, take **every cut within
1 SE of the best** as statistically tied, and choose the **median of that tied band**.
That lands in the middle of the plateau rather than on whichever noise spike happened to
top the list, so the choice is stable — and the printed band width tells you honestly how
well-determined the threshold really is.

Fitted on **val (134)**. Never on test.

> A wide tied band is not a bug — it's the BCE-saturation effect being reported instead of
> hidden. A head trained to push logits to ±∞ leaves a large empty region in the middle of
> its logit histogram, and *every* cut inside that gap produces the same mask.

In [ ]:
# Cut grid on the raw logit. Wide enough to cover a saturated head: sigmoid(+-12) is
# ~6e-6 / ~0.999994, far outside the [0.05, 0.95] the old probability-space sweep could
# reach. We check the winner isn't at an edge rather than trusting the range.
CUT_GRID = torch.linspace(-12.0, 12.0, 481, device=DEVICE)    # step 0.05


def val_logits_and_gt(entry):
    """Forward the whole val set once; cache bruise logits + GT on the GPU.

    134 x 640 x 640 floats ~= 219 MB, plus a 55 MB bool GT. Cheap, and it means the
    481-point sweep costs one forward pass over val, not 481 of them.
    """
    zs, gs = [], []
    for x, y, *_ in val_loader:
        zs.append(bruise_logit_640(entry, x.to(DEVICE, non_blocking=True)))
        gs.append((y[:, 0] > 0.5).to(DEVICE, non_blocking=True))
    return torch.cat(zs), torch.cat(gs)


def dice_per_image(z, gt, cut):
    """Per-image Dice at one cut. Matches pipeline.metrics.dice_np, including its
    empty-prediction-AND-empty-GT convention (denominator 0 -> Dice 1.0)."""
    pred = z >= cut
    inter = (pred & gt).sum(dim=(1, 2)).float()
    denom = pred.sum(dim=(1, 2)).float() + gt.sum(dim=(1, 2)).float()
    return torch.where(denom > 0, 2.0 * inter / denom, torch.ones_like(denom))


def sweep(entry):
    """1-D cut sweep on val. Returns the full grid plus the chosen operating point."""
    with torch.inference_mode():
        z, gt = val_logits_and_gt(entry)

        means = torch.empty(len(CUT_GRID), device=DEVICE)
        for i, c in enumerate(CUT_GRID):
            means[i] = dice_per_image(z, gt, c).mean()

        cuts = CUT_GRID.cpu().numpy()
        md = means.cpu().numpy()
        best_i = int(md.argmax())

        # Standard error of the mean Dice at the peak defines the tie band: any cut
        # scoring within 1 SE of the best is not distinguishable from it on 134 images.
        peak = dice_per_image(z, gt, CUT_GRID[best_i])
        se = float(peak.std(unbiased=True) / np.sqrt(peak.numel()))

        del z, gt
        torch.cuda.empty_cache()

    tied = cuts[md >= md[best_i] - se]
    chosen_cut = float(np.median(tied))
    chosen_thr = float(1.0 / (1.0 + np.exp(-chosen_cut)))

    if best_i in (0, len(cuts) - 1):
        print(f"  !! {entry['display']}: optimum sits at the EDGE of the cut grid "
              f"({cuts[best_i]:+.2f}). Widen CUT_GRID — the true optimum may be outside it.")

    grid = pd.DataFrame({'cut': cuts, 'threshold': 1.0 / (1.0 + np.exp(-cuts)), 'mean_dice': md})
    return {'grid': grid, 'cut': chosen_cut, 'threshold': chosen_thr, 'se': se,
            'argmax_cut': float(cuts[best_i]), 'peak_dice': float(md[best_i]),
            'tie_lo': float(tied.min()), 'tie_hi': float(tied.max()), 'n_tied': int(tied.size)}


print(f'Sweeping {len(CUT_GRID)} cuts on {len(val_df)} val images, per model.\n')
sweeps = {}
for run, m in MODELS.items():
    s = sweeps[run] = sweep(m)
    m['threshold'] = s['threshold']       # <-- the operating point everything below uses
    m['cut'] = s['cut']
    print(f"{m['display']:26s} cut={s['cut']:+.3f} -> threshold={s['threshold']:.4f} | "
          f"val peak Dice={s['peak_dice']:.4f}")
    print(f"{'':26s} tied band: {s['n_tied']:3d}/{len(CUT_GRID)} cuts within 1 SE "
          f"({s['se']:.2e}) -> cut in [{s['tie_lo']:+.2f}, {s['tie_hi']:+.2f}]")
    if m['csv_threshold'] is not None:
        print(f"{'':26s} historical CSV equiv threshold: {m['csv_threshold']:.4f} "
              f"(delta {s['threshold']-m['csv_threshold']:+.4f})")

### The sweep curve

The flat top is the point. Where the curve is flat, the threshold is genuinely
underdetermined by 134 val images, and any "best" pulled from that region is a coin
flip between equivalent options. The shaded band is the 1-SE tie region; the dashed
line is the median cut we selected from it.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.6 * len(MODELS), 3.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, (run, m) in zip(axes, MODELS.items()):
    s, g = sweeps[run], sweeps[run]['grid']
    ax.plot(g.cut, g.mean_dice, lw=1.4, color='#1f77b4')

    # The 1-SE tie band: every cut in here is statistically indistinguishable from
    # the peak on 134 images. Its width IS the honest uncertainty on the threshold.
    ax.axvspan(s['tie_lo'], s['tie_hi'], color='orange', alpha=0.18,
               label=f"1-SE tie band ({s['n_tied']} cuts)")
    ax.plot(s['argmax_cut'], s['peak_dice'], 'o', ms=5, color='k',
            label=f"argmax {s['argmax_cut']:+.2f}")
    ax.axvline(s['cut'], ls='--', c='crimson', lw=1.3,
               label=f"chosen {s['cut']:+.2f}")

    ax.set_title(m['display'], fontsize=9)
    ax.set_xlabel('cut on bruise logit')
    ax.set_xlim(-8, 8)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=6.5, loc='lower center')

axes[0].set_ylabel('val mean Dice')
plt.tight_layout()
plt.show()

---
# 11. Benchmark — 640 in → 640 out, on the GPU, over all 185 test images

### ✅ INCLUDED in the timer
- model forward pass
- bilinear upsample of logits to 640
- logit difference (YOLO) / channel select (SegFormer)
- sigmoid + threshold

### ❌ EXCLUDED from the timer
- disk read / decode
- resize to 640 + ImageNet normalisation (done once, by the dataloader, §6)
- host→device copy (done once, §7 — tensors are already resident)
- **any device→host copy** — the mask stays on the GPU
- model loading and warmup

### Two things that make GPU timing correct

* **`torch.cuda.synchronize()`** — the GPU is *asynchronous*. Without it we'd time how
  long it takes to *queue* the work (microseconds), not to *do* it (milliseconds).
* **Warmup** — the first calls pay one-off costs (cuDNN autotuning, allocator growth).

All 185 images are timed, `N_REPEATS` times each, so the spread is real across-image
variation rather than one image's noise. Median is the typical case; p95 is the "how bad
does it get" number a responsive UI actually cares about.

In [ ]:
N_WARMUP, N_REPEATS = 20, 3

def benchmark(entry):
    """Time bruise_mask_640 on every staged test image, N_REPEATS times each."""
    n = X_TEST.shape[0]
    with torch.inference_mode():
        for i in range(N_WARMUP):                       # steady state before we measure
            bruise_mask_640(entry, X_TEST[i % n:i % n + 1])
        torch.cuda.synchronize(DEVICE)

        # Everything already resident -- X_TEST (~0.9 GB) plus all 5 models' weights --
        # is not this model's inference cost, so subtract it and report only the
        # transient activation memory one forward pass actually adds.
        torch.cuda.reset_peak_memory_stats(DEVICE)
        baseline_b = torch.cuda.memory_allocated(DEVICE)

        lat = []
        for _ in range(N_REPEATS):
            for i in range(n):
                x = X_TEST[i:i + 1]                     # already on GPU, already 640
                torch.cuda.synchronize(DEVICE)          # GPU idle before we start
                t0 = time.perf_counter()
                bruise_mask_640(entry, x)               # 640 in -> 640 out, never leaves GPU
                torch.cuda.synchronize(DEVICE)          # wait for it to actually finish
                lat.append((time.perf_counter() - t0) * 1000.0)

    lat = np.asarray(lat)
    return {'mean_ms': lat.mean(), 'median_ms': np.median(lat),
            'p95_ms': np.percentile(lat, 95), 'std_ms': lat.std(),
            'fps': 1000.0 / lat.mean(), 'n_timed': int(lat.size),
            'activation_mb': (torch.cuda.max_memory_allocated(DEVICE) - baseline_b) / 1024**2}

print(f'Timing {len(test_df)} images x {N_REPEATS} repeats = '
      f'{len(test_df)*N_REPEATS} timed calls per model, after {N_WARMUP} warmup.\n')

bench = {}
for run, m in MODELS.items():
    bench[run] = b = benchmark(m)
    print(f"{m['display']:26s} {b['median_ms']:6.2f} ms (median) | p95 {b['p95_ms']:6.2f} | "
          f"{b['fps']:6.1f} FPS | {b['n_timed']} calls | +{b['activation_mb']:5.1f} MB act.")

---
# 12. Accuracy — all 5 models on all 185 test images at 640

Same `bruise_mask_640` the benchmark just timed, same tensors, same val-fitted
thresholds. The only additions are the GT comparison and the `.cpu()` numpy metrics
need — which is exactly why they live here and not in the timed loop.

Prediction and GT are both already on the dataloader's 640 grid, so **nothing is resized
to make the comparison**.

**Metrics.** Dice/IoU measure overlap. **Complete-miss rate** is the fraction of images
with a bruise where the model predicts *nothing* — reported separately because a blank
mask is a missed injury, far worse than a loose outline.

In [ ]:
per_image = {}
with torch.inference_mode():
    for run, m in MODELS.items():
        rows = []
        for i in range(X_TEST.shape[0]):
            pred = bruise_mask_640(m, X_TEST[i:i+1])[0].to(torch.uint8).cpu().numpy()
            gt   = (GT_640[i, 0].numpy() > 0.5).astype('uint8')
            rows.append(compute_image_row(pred, gt, STEMS[i]))
            if (i + 1) % 50 == 0:
                print(f"  [{m['display']}] {i+1}/{X_TEST.shape[0]}")

        df = pd.DataFrame(rows)
        df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
        per_image[run] = df
        print(f"{m['display']:26s} median Dice={df.dice.median():.3f}  mean={df.dice.mean():.3f}  "
              f"miss={df.complete_miss.mean()*100:.2f}%")

## Final results

One preprocessing recipe, one inference path, one sweep mechanism, one timing scope.
Thresholds fitted on val (134), everything reported here measured on test (185).

Read `complete_miss_%` next to `median_Dice` — the best-Dice model is not necessarily
the one you want in the field.

In [ ]:
results = pd.DataFrame([{
    'model': m['display'],
    'params_M': round(m['params_m'], 2),
    # --- fitted on val ---
    'cut': round(m['cut'], 3),
    'threshold': round(m['threshold'], 4),
    'tie_band': f"[{sweeps[run]['tie_lo']:+.2f}, {sweeps[run]['tie_hi']:+.2f}]",
    'val_peak_Dice': round(sweeps[run]['peak_dice'], 4),
    # --- measured on test ---
    'median_Dice': round(per_image[run].dice.median(), 4),
    'mean_Dice': round(per_image[run].dice.mean(), 4),
    'median_IoU': round(per_image[run].iou.median(), 4),
    'complete_miss_%': round(per_image[run].complete_miss.mean() * 100, 2),
    'median_ms': round(bench[run]['median_ms'], 2),
    'p95_ms': round(bench[run]['p95_ms'], 2),
    'FPS': round(bench[run]['fps'], 1),
    'activation_MB': round(bench[run]['activation_mb'], 1),
} for run, m in MODELS.items()])
results

---
## YOLO native argmax pass

The demo scored YOLO through custom /255. We also score native `.predict()` argmax (YOLO's
home turf) so the analysis can use it as the core YOLO number and compare the two paths.

In [ ]:
from ultralytics import YOLO as _YOLO_native
def _to640_nn(mask):
    m = np.asarray(mask)
    if m.ndim == 3: m = m[..., 0]
    return (cv2.resize(m.astype('uint8'), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) > 0).astype('uint8')
native_per_image = {}
for run in ['yolo_sem_direct', 'yolo_sem_distilled']:
    w = _YOLO_native(MODELS[run]['ckpt']); rows = []
    for i, (_, r) in enumerate(test_df.iterrows()):
        res = w.predict(source=str(r.image_path), imgsz=IMG_H, device=0, verbose=False)[0]
        if getattr(res, 'semantic_mask', None) is not None:
            cm = res.semantic_mask.data; cm = cm.cpu().numpy() if hasattr(cm, 'cpu') else np.asarray(cm)
            pred = _to640_nn((cm == 1).astype('uint8'))
        else:
            pred = np.zeros((IMG_H, IMG_W), np.uint8)
        gt = (GT_640[i, 0].numpy() > 0.5).astype('uint8')
        rows.append(compute_image_row(pred, gt, STEMS[i]))
    df = pd.DataFrame(rows); df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    native_per_image[run] = df
    print(f"{MODELS[run]['display']:26s} native argmax median={df.dice.median():.3f} miss={df.complete_miss.mean()*100:.2f}%")

## Skin-tone (ITA) + inter-labeler labels (embedded — not in the zip)

In [ ]:
ITA_CSV_TEXT = """stem,subject,skin_tone_category,ita_group_index_5
TAM006_V5_UA_N,TAM006,Dark (VI),4
TAM006_V17_UA_N,TAM006,Dark (VI),4
TAM0009_V2_UA_N,TAM0009,Intermediate (III-IV),1
TAM009_V4_UA_N,TAM009,Intermediate (III-IV),1
TAM028_V13_UA_N,TAM028,Light (II-III),0
TAM030_V10_UA_N,TAM030,Tan (IV),2
TAM030_V11_UA_N,TAM030,Intermediate (III-IV),1
TAM030_V12_UA_N,TAM030,Brown (V),3
TAM030_V13_UA_N,TAM030,Intermediate (III-IV),1
TAM031_V1_UA_N,TAM031,Tan (IV),2
TAM031_V2_UA_N,TAM031,Intermediate (III-IV),1
TAM031_V4_UA_N,TAM031,Tan (IV),2
TAM033_V10_UA_N,TAM033,Tan (IV),2
TAM033_V11_UA_N,TAM033,Tan (IV),2
TAM033_V13_UA_N,TAM033,Tan (IV),2
TAM033_V15_UA_N,TAM033,Dark (VI),4
TAM034_V4_UA_N,TAM034,Intermediate (III-IV),1
TAM034_V5_UA_N,TAM034,Intermediate (III-IV),1
TAM034_V9_LA_N,TAM034,Light (II-III),0
TAM034_V11_UA_N,TAM034,Intermediate (III-IV),1
TAM034_V14_UA_N,TAM034,Intermediate (III-IV),1
TAM034_V15_UA_N,TAM034,Intermediate (III-IV),1
TAM034_V17_UA_N,TAM034,Intermediate (III-IV),1
TAM038_V1_UA_N,TAM038,Dark (VI),4
TAM038_V2_UA_N,TAM038,Intermediate (III-IV),1
TAM038_V3_UA_N,TAM038,Tan (IV),2
TAM038_V5_UA_N,TAM038,Intermediate (III-IV),1
TAM038_V11_UA_N,TAM038,Dark (VI),4
TAM038_V12_UA_N,TAM038,Brown (V),3
TAM047_V12_UA_N,TAM047,Tan (IV),2
TAM047_V15_UA_N,TAM047,Brown (V),3
TAM047_V16_UA_N,TAM047,Intermediate (III-IV),1
TAM047_V19_UA_N,TAM047,Intermediate (III-IV),1
TAM047_V20_UA_N,TAM047,Tan (IV),2
TAM057_V9_UA_N,TAM057,Intermediate (III-IV),1
TAM057_V10_UA_N,TAM057,Tan (IV),2
TAM057_V11_LA_N,TAM057,Intermediate (III-IV),1
TAM057_V11_UA_N,TAM057,Intermediate (III-IV),1
TAM057_V12_LA_N,TAM057,Light (II-III),0
TAM057_V13_LA_N,TAM057,Light (II-III),0
TAM057_V13_UA_N,TAM057,Intermediate (III-IV),1
TAM057_V14_LA_N,TAM057,Light (II-III),0
TAM057_V14_UA_N,TAM057,Dark (VI),4
TAM057_V15_UA_N,TAM057,Light (II-III),0
TAM058_V1_UA_N,TAM058,Light (II-III),0
TAM058_V2_UA_N,TAM058,Light (II-III),0
TAM058_V3_UA_N,TAM058,Light (II-III),0
TAM058_V4_UA_N,TAM058,Light (II-III),0
TAM058_V5_UA_N,TAM058,Light (II-III),0
TAM058_V6_UA_N,TAM058,Light (II-III),0
TAM058_V8_UA_N,TAM058,Light (II-III),0
TAM058_V9_UA_N,TAM058,Light (II-III),0
TAM058_V10_UA_N,TAM058,Light (II-III),0
TAM058_V12_UA_N,TAM058,Light (II-III),0
TAM058_V13_UA_N,TAM058,Intermediate (III-IV),1
TAM058_V14_UA_N,TAM058,Light (II-III),0
TAM058_V15_UA_N,TAM058,Light (II-III),0
TAM058_V19_UA_N,TAM058,Light (II-III),0
TAM063_V1_UA_N,TAM063,Brown (V),3
TAM063_V2_UA_N,TAM063,Brown (V),3
TAM063_V3_UA_N,TAM063,Tan (IV),2
TAM063_V4_UA_N,TAM063,Dark (VI),4
TAM063_V5_UA_N,TAM063,Brown (V),3
TAM063_V6_UA_N,TAM063,Brown (V),3
TAM063_V7_UA_N,TAM063,Brown (V),3
TAM063_V8_UA_N,TAM063,Intermediate (III-IV),1
TAM063_V10_UA_N,TAM063,Dark (VI),4
TAM063_V12_UA_N,TAM063,Tan (IV),2
TAM063_V13_UA_N,TAM063,Dark (VI),4
TAM063_V15_UA_N,TAM063,Brown (V),3
TAM063_V16_UA_N,TAM063,Brown (V),3
TAM063_V17_UA_N,TAM063,Tan (IV),2
TAM063_V18_UA_N,TAM063,Brown (V),3
TAM063_V19_UA_N,TAM063,Dark (VI),4
TAM063_V20_UA_N,TAM063,Dark (VI),4
TAM063_V21_UA_N,TAM063,Dark (VI),4
TAM066_V20_UA_N,TAM066,Light (II-III),0
TAM067_V1_UA_N,TAM067,Dark (VI),4
TAM067_V2_UA_N,TAM067,Dark (VI),4
TAM067_V3_UA_N,TAM067,Dark (VI),4
TAM067_V4_UA_N,TAM067,Brown (V),3
TAM067_V5_UA_N,TAM067,Tan (IV),2
TAM067_V6_UA_N,TAM067,Intermediate (III-IV),1
TAM067_V7_UA_N,TAM067,Brown (V),3
TAM067_V8_UA_N,TAM067,Dark (VI),4
TAM067_V9_UA_N,TAM067,Dark (VI),4
TAM067_V10_UA_N,TAM067,Dark (VI),4
TAM067_V11_UA_N,TAM067,Dark (VI),4
TAM069_V9_UA_N,TAM069,Dark (VI),4
TAM076_V3_UA_N,TAM076,Dark (VI),4
TAM076_V4_UA_N,TAM076,Dark (VI),4
TAM076_V5_UA_N,TAM076,Dark (VI),4
TAM076_V6_UA_N,TAM076,Brown (V),3
TAM076_V7_UA_N,TAM076,Dark (VI),4
TAM076_V8_UA_N,TAM076,Brown (V),3
TAM076_V9_UA_N,TAM076,Dark (VI),4
TAM076_V10_UA_N,TAM076,Dark (VI),4
TAM076_V11_UA_N,TAM076,Dark (VI),4
TAM076_V12_UA_N,TAM076,Dark (VI),4
TAM077_V17_UA_N,TAM077,Intermediate (III-IV),1
TAM077_V18_UA_N,TAM077,Tan (IV),2
TAM077_V19_UA_N,TAM077,Brown (V),3
TAM077_V20_UA_N,TAM077,Dark (VI),4
TAM077_V21_UA_N,TAM077,Intermediate (III-IV),1
TAM079_V4_UA_N,TAM079,Light (II-III),0
TAM079_V5_UA_N,TAM079,Tan (IV),2
TAM079_V6_UA_N,TAM079,Light (II-III),0
TAM079_V7_UA_N,TAM079,Intermediate (III-IV),1
TAM079_V8_UA_N,TAM079,Light (II-III),0
TAM079_V9_UA_N,TAM079,Intermediate (III-IV),1
TAM079_V10_UA_N,TAM079,Intermediate (III-IV),1
TAM079_V11_UA_N,TAM079,Intermediate (III-IV),1
TAM079_V12_UA_N,TAM079,Light (II-III),0
TAM080_V2_UA_N,TAM080,Dark (VI),4
TAM080_V3_UA_N,TAM080,Dark (VI),4
TAM080_V4_UA_N,TAM080,Tan (IV),2
TAM080_V5_UA_N,TAM080,Brown (V),3
TAM080_V6_UA_N,TAM080,Dark (VI),4
TAM080_V7_UA_N,TAM080,Brown (V),3
TAM080_V8_UA_N,TAM080,Brown (V),3
TAM080_V10_UA_N,TAM080,Tan (IV),2
TAM080_V14_UA_N,TAM080,Brown (V),3
TAM080_V17_UA_N,TAM080,Brown (V),3
TAM081_V10_UA_N,TAM081,Dark (VI),4
TAM081_V11_UA_N,TAM081,Dark (VI),4
TAM081_V12_UA_N,TAM081,Dark (VI),4
TAM082_V1_UA_N,TAM082,Brown (V),3
TAM082_V2_UA_N,TAM082,Dark (VI),4
TAM082_V3_UA_N,TAM082,Brown (V),3
TAM082_V5_UA_N,TAM082,Brown (V),3
TAM082_V6_UA_N,TAM082,Brown (V),3
TAM083_V1_UA_N,TAM083,Light (II-III),0
TAM083_V2_UA_N,TAM083,Intermediate (III-IV),1
TAM083_V3_UA_N,TAM083,Brown (V),3
TAM083_V4_UA_N,TAM083,Intermediate (III-IV),1
TAM083_V5_UA_N,TAM083,Tan (IV),2
TAM083_V8_UA_N,TAM083,Brown (V),3
TAM083_V9_UA_N,TAM083,Tan (IV),2
TAM084_V2_UA_N,TAM084,Light (II-III),0
TAM084_V3_UA_N,TAM084,Light (II-III),0
TAM084_V5_UA_N,TAM084,Light (II-III),0
TAM084_V6_UA_N,TAM084,Intermediate (III-IV),1
TAM084_V7_UA_N,TAM084,Tan (IV),2
TAM084_V9_UA_N,TAM084,Light (II-III),0
TAM084_V10_UA_N,TAM084,Intermediate (III-IV),1
TAM085_V1_UA_N,TAM085,Light (II-III),0
TAM085_V3_UA_N,TAM085,Light (II-III),0
TAM085_V4_UA_N,TAM085,Light (II-III),0
TAM085_V12_UA_N,TAM085,Intermediate (III-IV),1
TAM085_V13_LA_N,TAM085,Intermediate (III-IV),1
TAM085_V13_UA_N,TAM085,Intermediate (III-IV),1
TAM085_V15_LA_N,TAM085,Tan (IV),2
TAM085_V15_UA_N,TAM085,Light (II-III),0
TAM085_V16_LA_N,TAM085,Intermediate (III-IV),1
TAM085_V16_UA_N,TAM085,Light (II-III),0
TAM085_V17_LA_N,TAM085,Intermediate (III-IV),1
TAM085_V17_UA_N,TAM085,Light (II-III),0
TAM085_V18_UA_N,TAM085,Light (II-III),0
TAM085_V19_LA_N,TAM085,Light (II-III),0
TAM085_V19_UA_N,TAM085,Light (II-III),0
TAM085_V21_LA_N,TAM085,Light (II-III),0
TAM086_V3_UA_N,TAM086,Intermediate (III-IV),1
TAM086_V4_UA_N,TAM086,Dark (VI),4
TAM086_V5_UA_N,TAM086,Tan (IV),2
TAM086_V6_UA_N,TAM086,Brown (V),3
TAM086_V7_UA_N,TAM086,Dark (VI),4
TAM086_V8_UA_N,TAM086,Tan (IV),2
TAM086_V9_UA_N,TAM086,Dark (VI),4
TAM086_V10_UA_N,TAM086,Dark (VI),4
TAM088_V1_UA_N,TAM088,Dark (VI),4
TAM088_V2_UA_N,TAM088,Dark (VI),4
TAM088_V3_UA_N,TAM088,Dark (VI),4
TAM088_V5_UA_N,TAM088,Dark (VI),4
TAM088_V11_UA_N,TAM088,Dark (VI),4
TAM088_V12_UA_N,TAM088,Dark (VI),4
TAM088_V13_UA_N,TAM088,Dark (VI),4
TAM088_V15_UA_N,TAM088,Dark (VI),4
TAM088_V16_UA_N,TAM088,Dark (VI),4
TAM088_V18_UA_N,TAM088,Dark (VI),4
TAM088_V19_UA_N,TAM088,Dark (VI),4
TAM088_V20_UA_N,TAM088,Dark (VI),4
TAM100_V10_UA_N,TAM100,Dark (VI),4
TAM100_V15_UA_N,TAM100,Dark (VI),4
TAM100_V16_UA_N,TAM100,Dark (VI),4
TAM100_V17_UA_N,TAM100,Brown (V),3
"""

import io as _io
ita = pd.read_csv(_io.StringIO(ITA_CSV_TEXT))
assert len(ita) == 185 and ita.subject.nunique() == 28
print("ITA:", ita.shape, "| subjects:", ita.subject.nunique())

In [ ]:
IL_CSV_TEXT = """stem,subject,paul_vs_gbarimah,paul_vs_erik,gbarimah_vs_erik,paul_vs_majority,gbarimah_vs_majority,erik_vs_majority
TAM006_V5_UA_N,TAM006,0.5500419011133725,0.6352281417282691,0.8061078912171199,0.6915498820591092,0.8548127836611196,0.9365378885688748
TAM006_V17_UA_N,TAM006,0.7119110748521313,0.7104420243433697,0.8494061757719715,0.7714301011939819,0.9177241584748288,0.9273813266268468
TAM0009_V2_UA_N,TAM0009,0.6733453769091805,0.7021750299708854,0.8907128257537047,0.7405734928877851,0.9265472907876816,0.958386309748806
TAM009_V4_UA_N,TAM009,0.5241364665911665,0.7519676884838442,0.7071249771048294,0.7741362187026293,0.7198944040273813,0.9801548681927812
TAM028_V13_UA_N,TAM028,0.7890300368039891,0.8572829551336184,0.8769237692376923,0.8849844501741049,0.899506129288297,0.977171464330413
TAM030_V10_UA_N,TAM030,0.9197130334712666,0.9348584428715876,0.9367713552486864,0.9589429924266212,0.9600514994352066,0.9763692662563151
TAM030_V11_UA_N,TAM030,0.9259305898352894,0.932777433272202,0.8918432725480532,0.985366827482948,0.9405933829785912,0.9480331141762752
TAM030_V12_UA_N,TAM030,0.2487487880931792,0.5681369321922317,0.4791643711849093,0.6078048468162756,0.4909626652517019,0.9522433160349472
TAM030_V13_UA_N,TAM030,0.2425306628995418,0.7888415005657585,0.3433186892509116,0.7952380952380952,0.3458302784784959,0.9955969395120544
TAM031_V1_UA_N,TAM031,0.4333745066730937,0.4461084638075788,0.4809464689987585,0.7501368862931191,0.6309942213271202,0.814404432132964
TAM031_V2_UA_N,TAM031,0.1559404963895355,0.6739922059669073,0.282923629207136,0.7155902152902753,0.2897449609972892,0.9564447223090548
TAM031_V4_UA_N,TAM031,0.1680514073671825,0.0,0.3083721733126187,0.4683921093789415,0.4338681838522022,0.793743890518084
TAM033_V10_UA_N,TAM033,0.7132611637347768,0.8435745227017487,0.6442652857058789,0.9664606707865844,0.7540939776752891,0.8746822033898305
TAM033_V11_UA_N,TAM033,0.0,0.845200776235259,0.0,0.9344005280963776,0.0,0.8985162262953265
TAM033_V13_UA_N,TAM033,0.7564356435643564,0.7502241056148643,0.8075855062648154,0.855257592420269,0.8961229479566888,0.8993757431629013
TAM033_V15_UA_N,TAM033,0.7898099882293593,0.7665882834066047,0.7160661297159814,0.9293943870014773,0.8534801544909195,0.844272076372315
TAM034_V4_UA_N,TAM034,0.7205001472946876,0.8248251227861289,0.8238873880278687,0.8632362635232076,0.8529256411761253,0.9611551491629116
TAM034_V5_UA_N,TAM034,0.640815714499925,0.8298223549756327,0.7888618690040352,0.8356341167729068,0.7929583466496218,0.9957162613417884
TAM034_V9_LA_N,TAM034,0.0,0.3693552010784093,0.0,0.4905998209489705,0.0,0.5991253644314869
TAM034_V11_UA_N,TAM034,0.9549406416876296,0.9406260910043058,0.9423307335820053,0.9769520756676224,0.977594417257346,0.9645964457608294
TAM034_V14_UA_N,TAM034,0.8286743268736065,0.8855640322939041,0.8737602853299127,0.922765933101949,0.9047466324567032,0.9633100217039308
TAM034_V15_UA_N,TAM034,0.8315668862790031,0.9069366877185656,0.8918764049896416,0.9241789144160876,0.9061899441340782,0.982805517777346
TAM034_V17_UA_N,TAM034,0.628761939088124,0.6029528676888132,0.7649516441005803,0.7529944966008417,0.8632255172928961,0.8912024517757346
TAM038_V1_UA_N,TAM038,0.2662786467806475,0.7387729285262492,0.4596345130696275,0.74749825459623,0.4632601515445891,0.9889583899918412
TAM038_V2_UA_N,TAM038,0.3008553365563406,0.2189605834025662,0.8768211920529801,0.3100804906094289,0.974272391059656,0.8992409109069117
TAM038_V3_UA_N,TAM038,0.8266438356164384,0.6400917431192661,0.6980121703853955,0.8983252698176405,0.9216016150740244,0.7708781362007169
TAM038_V5_UA_N,TAM038,0.871778060074581,0.9041923306179216,0.8223464554701452,0.9800546221964744,0.893089381639,0.9244151874926532
TAM038_V11_UA_N,TAM038,0.2945632048476687,0.4956103086944208,0.687904687904688,0.4956103086944208,0.687904687904688,1.0
TAM038_V12_UA_N,TAM038,0.790930916976456,0.8738059101865905,0.8766231430524082,0.8949305525128161,0.8936645170434413,0.9827016129032258
TAM047_V12_UA_N,TAM047,0.9029574051907862,0.9361786939623876,0.8824130100135648,0.9761738693747734,0.9263280341217526,0.9596604206945676
TAM047_V15_UA_N,TAM047,0.2899713698132989,0.7397006634778583,0.4387932453941711,0.7497263201793254,0.4408904575451755,0.98927569707969
TAM047_V16_UA_N,TAM047,0.783801652892562,0.9237730595196658,0.85664,0.9259839777081156,0.8586295228946526,0.9978121844496802
TAM047_V19_UA_N,TAM047,0.800732516882225,0.9034112207922028,0.8176656151419558,0.9488188976377953,0.851977648734524,0.954366734183248
TAM047_V20_UA_N,TAM047,0.8201503421967912,0.7359893252001525,0.849268167422751,0.8553389455028799,0.9709364908503768,0.8759111713849805
TAM057_V9_UA_N,TAM057,0.7276883420067117,0.7342844621901933,0.9163974773111828,0.7725014087393064,0.9536577557227468,0.9593013940073706
TAM057_V10_UA_N,TAM057,0.3902344880826016,0.3614587057561673,0.943908151025304,0.3932130876561816,0.9952720811877014,0.9481596236050148
TAM057_V11_LA_N,TAM057,0.5098270440251572,0.7013566251626092,0.7565606998079795,0.7191650853889943,0.7784090909090909,0.9739166153214212
TAM057_V11_UA_N,TAM057,0.3537377619178333,0.3249558712280407,0.8624472573839662,0.3824314966861212,0.9522478453296064,0.9014332965821388
TAM057_V12_LA_N,TAM057,0.6892655367231638,0.4897172236503856,0.6872825666795516,0.7668009669621273,0.9098424179798502,0.755203171456888
TAM057_V13_LA_N,TAM057,0.277526395173454,0.6282076785359061,0.4658261446582614,0.8217538381472808,0.6311057836379982,0.7927170868347339
TAM057_V13_UA_N,TAM057,0.3665121352604308,0.3425670594133868,0.9018204944749184,0.387055657838833,0.9727319499228264,0.9254835039817976
TAM057_V14_LA_N,TAM057,0.057003823427181,0.1547319179560993,0.7698713096139288,0.1618366578848325,0.8071428571428572,0.9496284062758052
TAM057_V14_UA_N,TAM057,0.3332009531374106,0.350488985478613,0.887625544817131,0.3856334720888035,0.9363268554752272,0.9458619777895292
TAM057_V15_UA_N,TAM057,0.2459016393442623,0.2281167108753315,0.8274967574578469,0.2798839277356548,0.9192213278529427,0.8931767337807607
TAM058_V1_UA_N,TAM058,0.4966099526672636,0.4944005012138773,0.6525399943867527,0.6989205646277332,0.8844087299700443,0.7584289639566834
TAM058_V2_UA_N,TAM058,0.7482643353047056,0.7150687765022236,0.7868706039968739,0.8314093314093314,0.926880589163598,0.8575039494470774
TAM058_V3_UA_N,TAM058,0.5608377659574468,0.7133928571428572,0.6437737642585551,0.8524939983995732,0.7791199309749784,0.8522933926045666
TAM058_V4_UA_N,TAM058,0.470927963786779,0.686970773012838,0.7135690540948927,0.7523182769967095,0.772998101224383,0.9256314312441534
TAM058_V5_UA_N,TAM058,0.5017103762827823,0.5970805838832234,0.7025158440560784,0.7295382360127046,0.8508955571063038,0.8432320441988951
TAM058_V6_UA_N,TAM058,0.3924237268912328,0.3763419080413206,0.5168004757656854,0.5793576551294044,0.841442749939482,0.6359427609427609
TAM058_V8_UA_N,TAM058,0.7749873928391326,0.5903170635442705,0.6724896239856284,0.8498106060606061,0.943917920180854,0.7234929711698832
TAM058_V9_UA_N,TAM058,0.4563419117647059,0.5405724402478608,0.6691216584833606,0.7058370256212676,0.8539599651871193,0.8057765267409129
TAM058_V10_UA_N,TAM058,0.6524376020527175,0.5257792262053982,0.7006807495262826,0.741369152821963,0.9271055808338752,0.7542719954808643
TAM058_V12_UA_N,TAM058,0.9257755400357316,0.7294423148262644,0.7664469742360696,0.9442932607651672,0.9832436587240584,0.7818612922347362
TAM058_V13_UA_N,TAM058,0.9000211371803002,0.8991446190570918,0.835186656076251,0.9843391104614744,0.9170487703209672,0.9151943462897526
TAM058_V14_UA_N,TAM058,0.2756009842892296,0.4136135391482239,0.8348457350272233,0.4380539688792594,0.868626922039385,0.960042621204049
TAM058_V15_UA_N,TAM058,0.3904939422180801,0.4482069057459005,0.603384702656708,0.6348187311178247,0.7623029144768275,0.766295707472178
TAM058_V19_UA_N,TAM058,0.0,0.1658031088082901,0.5922073097106766,0.2029671038486347,0.655697243932538,0.8753363228699551
TAM063_V1_UA_N,TAM063,0.6466004285992597,0.4865142187041923,0.7533580361278369,0.6756234096692112,0.9672192076116852,0.7730972610064761
TAM063_V2_UA_N,TAM063,0.8210220392346815,0.575828985019357,0.7274778162129644,0.8254312944866691,0.9968057702215352,0.7304373921524495
TAM063_V3_UA_N,TAM063,0.7310937260315999,0.7410346854791299,0.7943485086342229,0.847679892400807,0.8859259259259259,0.8889753566796368
TAM063_V4_UA_N,TAM063,0.7676767676767676,0.703607088243231,0.7394711911118766,0.8766110052672867,0.9117553360942976,0.8204288754296939
TAM063_V5_UA_N,TAM063,0.6356388279848515,0.813752986392438,0.7805910746041155,0.8457303249487207,0.8054172127566623,0.9670230114151116
TAM063_V6_UA_N,TAM063,0.7609097484529455,0.662306032983091,0.8259713228492137,0.7980212962186636,0.961890654457312,0.8543504171632896
TAM063_V7_UA_N,TAM063,0.7040200432752534,0.6758313959365208,0.7639305972639306,0.8179593834108148,0.8918103028532477,0.8489587909674018
TAM063_V8_UA_N,TAM063,0.7002407704654896,0.6131742168485514,0.8096676737160121,0.7545893719806763,0.9641829229293252,0.843752045558683
TAM063_V10_UA_N,TAM063,0.7997416020671835,0.8086095144918675,0.8530339805825242,0.8831897415347626,0.9310550367574012,0.9211802979282004
TAM063_V12_UA_N,TAM063,0.8653027598517095,0.7125279642058165,0.8318278696843601,0.8705753724203908,0.9958687727825032,0.8358450490543138
TAM063_V13_UA_N,TAM063,0.8591874931190135,0.7619355359883674,0.7254248583805398,0.9530160654743862,0.9145137576139466,0.8063660477453581
TAM063_V15_UA_N,TAM063,0.8439546730429358,0.6367611979856878,0.6375736706178399,0.9271587071876508,0.928089454405244,0.7017101028585853
TAM063_V16_UA_N,TAM063,0.8415234476206926,0.7062248717593235,0.8196244547006386,0.8635361925750127,0.9829403290620972,0.8361187328510851
TAM063_V17_UA_N,TAM063,0.8410372040586246,0.6274972253052165,0.7644561009117022,0.8461322607800954,0.9961754414517048,0.7680717265425565
TAM063_V18_UA_N,TAM063,0.8352930110055498,0.5986230221041189,0.6569427562655553,0.8892078586166682,0.956514410921962,0.6958916427808689
TAM063_V19_UA_N,TAM063,0.9007779619946436,0.7779110444777612,0.7848906560636183,0.9490306060236556,0.9563953488372092,0.8268714011516315
TAM063_V20_UA_N,TAM063,0.8935700672083459,0.6734892787524367,0.6756256427836819,0.950282706180542,0.9474186468174444,0.7189727040473309
TAM063_V21_UA_N,TAM063,0.8546091015169195,0.6476911936723667,0.7342022940563087,0.8825136612021858,0.9777129764632368,0.7550382031690682
TAM066_V20_UA_N,TAM066,0.6677862412283332,0.5235219195529808,0.7288291475061903,0.7290724952795102,0.9624439461883408,0.7639815642842402
TAM067_V1_UA_N,TAM067,0.5334781542342595,0.4942757009345794,0.8517005386557027,0.5758421231711467,0.9747163405699992,0.8765212078008046
TAM067_V2_UA_N,TAM067,0.5509926041261192,0.5578504494167145,0.8859060402684564,0.6075184838071436,0.9398362406550372,0.9399030013354888
TAM067_V3_UA_N,TAM067,0.7909078074262318,0.7574300699300699,0.7774072081764389,0.8946824987093444,0.9157267773412748,0.8590361445783132
TAM067_V4_UA_N,TAM067,0.7784538144066715,0.6160065199674002,0.7912816733745253,0.7970641582654912,0.986013986013986,0.8036408724567435
TAM067_V5_UA_N,TAM067,0.6328594765562545,0.6453533706670447,0.9447352812234056,0.6640166295904598,0.9644587194608256,0.9788120875303924
TAM067_V6_UA_N,TAM067,0.5835044958692774,0.6601951720595789,0.8599291402111762,0.6863519863306279,0.8829101386637853,0.970702045328911
TAM067_V7_UA_N,TAM067,0.6819994221323317,0.6525850152059718,0.9293708665423096,0.6999169730755544,0.980192264699307,0.9471592136530568
TAM067_V8_UA_N,TAM067,0.5438482655353132,0.5815542271562767,0.8713183856502242,0.6193724420190996,0.9086079832404323,0.9550173010380624
TAM067_V9_UA_N,TAM067,0.7478877078222949,0.5530028214429665,0.7559028367589604,0.7628579371698637,0.984077841662981,0.765262252794497
TAM067_V10_UA_N,TAM067,0.4967925475539689,0.4421167593328038,0.7284073375621464,0.5907794361525704,0.8816401062416999,0.8073575799034698
TAM067_V11_UA_N,TAM067,0.4497542254816917,0.506977984532875,0.7865842507589335,0.569176008782143,0.8434717809938425,0.9210278372591006
TAM069_V9_UA_N,TAM069,0.5519785378940308,0.4629085682144866,0.8797074456003271,0.55333930972658,0.9982987936900712,0.8810282074613285
TAM076_V3_UA_N,TAM076,0.5566588785046729,0.7181612660135644,0.7778643336529243,0.7400145595729192,0.7927570307490305,0.9764188198127044
TAM076_V4_UA_N,TAM076,0.5898916537683058,0.5949314499376818,0.8261216070507712,0.6869800332778702,0.8886125838690001,0.9295774647887324
TAM076_V5_UA_N,TAM076,0.3956061078217513,0.3195647030991246,0.831082371558757,0.4614475896224272,0.9023850766155896,0.9233660725112162
TAM076_V6_UA_N,TAM076,0.4159033454808102,0.4418401869484845,0.8670311222073953,0.4784710681371808,0.9100232678970792,0.9483274088373078
TAM076_V7_UA_N,TAM076,0.2660106216807247,0.3828687050359712,0.7863780474472505,0.3828687050359712,0.7863780474472505,1.0
TAM076_V8_UA_N,TAM076,0.326438759638614,0.4244697361614071,0.8287216698431652,0.4291307752545027,0.8331525877385265,0.993722121262184
TAM076_V9_UA_N,TAM076,0.2763844684914067,0.4035691049354029,0.7281601663126497,0.4245624327759851,0.7460986339430423,0.9680365296803652
TAM076_V10_UA_N,TAM076,0.4795500357630535,0.4606064391218811,0.8972122171097833,0.5099923933337943,0.9591364344535944,0.932871522892088
TAM076_V11_UA_N,TAM076,0.5209700948212983,0.3518256265008312,0.7084903465922308,0.54567158477773,0.9684270770404756,0.7252422792104198
TAM076_V12_UA_N,TAM076,0.5073336984967439,0.5566796888043006,0.8169661241032532,0.6133245042857668,0.8690660088073225,0.9316361838853188
TAM077_V17_UA_N,TAM077,0.93631156930126,0.7312916111850866,0.7843837713702475,0.9401963022141064,0.9965420358763778,0.7876834337093901
TAM077_V18_UA_N,TAM077,0.8051345927617466,0.66796875,0.8418487891317188,0.8118764191839263,0.9952283968577248,0.8465045592705167
TAM077_V19_UA_N,TAM077,0.7249719438102241,0.581904414447282,0.8190841815472432,0.7364937663537017,0.9920768454220074,0.8262707599396074
TAM077_V20_UA_N,TAM077,0.7429759429759429,0.6500671556979026,0.8490829768308695,0.7695224117898857,0.9729178800232964,0.8700652916504519
TAM077_V21_UA_N,TAM077,0.7345663789560214,0.7690024732069249,0.8592236253090132,0.8262178919397697,0.9103135839173214,0.9403917116094238
TAM079_V4_UA_N,TAM079,0.7817251461988304,0.674062401512764,0.8577794281470479,0.7961584276354973,0.9848946278474844,0.8692146427995484
TAM079_V5_UA_N,TAM079,0.8698289917718879,0.832225237449118,0.897898767577308,0.904079301322917,0.971114359295608,0.9264156276498854
TAM079_V6_UA_N,TAM079,0.805848080193971,0.8380565277934591,0.9194013026052104,0.862599263993802,0.9416933932007696,0.9748987316554883
TAM079_V7_UA_N,TAM079,0.7830139437774961,0.8224467418546366,0.9436152570480928,0.8302826645582131,0.9509239916439016,0.9919206141674208
TAM079_V8_UA_N,TAM079,0.9471613863491388,0.9173402868318122,0.9386658707655574,0.9636863972373604,0.9835054724834283,0.953995157384988
TAM079_V9_UA_N,TAM079,0.8654007975707021,0.764069613509412,0.8949769053117783,0.8656083107762089,0.9998387460895928,0.8951363833164959
TAM079_V10_UA_N,TAM079,0.8642505592841163,0.8670263684622749,0.8860658991663358,0.9252414561664192,0.9400269541778976,0.9406793567443388
TAM079_V11_UA_N,TAM079,0.7850826889637057,0.8007306066900651,0.8834412297119013,0.8551539491298528,0.9343847352024922,0.943959128323189
TAM079_V12_UA_N,TAM079,0.7825932504440497,0.7627236006924408,0.9415293781544616,0.8012851600387972,0.9804641015470053,0.9595302795031057
TAM080_V2_UA_N,TAM080,0.8144646263581322,0.7563154651879236,0.9318266430151326,0.8182196917094848,0.996113868339732,0.9352264984504473
TAM080_V3_UA_N,TAM080,0.8513500620169832,0.7468544287976002,0.8836452400325467,0.8556971693159579,0.9961225766103816,0.8870305272895467
TAM080_V4_UA_N,TAM080,0.8384833674917231,0.815052776502983,0.8933138956994235,0.8819177516015706,0.95603811683021,0.93177038441648
TAM080_V5_UA_N,TAM080,0.8231403040501335,0.7374870197300104,0.9061462539255272,0.8253341080766996,0.9979249011857708,0.907931375190874
TAM080_V6_UA_N,TAM080,0.9019191024505462,0.9284704551880124,0.9254183982927104,0.9537214016796988,0.9504198780628093,0.9743052492280548
TAM080_V7_UA_N,TAM080,0.8767726161369194,0.7251375654274594,0.8391787852865698,0.8789653489507077,0.9982555604012212,0.8408785845027456
TAM080_V8_UA_N,TAM080,0.8466298993811308,0.8368644960195443,0.8596811933313835,0.9163368363586734,0.92950922163176,0.920484753713839
TAM080_V10_UA_N,TAM080,0.8802678039161737,0.8810227062949972,0.8608497409326424,0.9548011447744992,0.9254059763235488,0.9264080765143464
TAM080_V14_UA_N,TAM080,0.5248306545352519,0.2080225193525686,0.5466170436110316,0.555136303896214,0.954566682233638,0.5649750949351482
TAM080_V17_UA_N,TAM080,0.6991915249512127,0.5928490136570561,0.8312502360004531,0.7250231267345051,0.9718390551319906,0.8518031264510137
TAM081_V10_UA_N,TAM081,0.4375998677613091,0.4378169790518192,0.8513070166155149,0.4952297811311342,0.9195244627343392,0.9198399085191538
TAM081_V11_UA_N,TAM081,0.4291939594924219,0.450868050557107,0.9062994960403168,0.474444814736268,0.9353568376862206,0.9668586789554532
TAM081_V12_UA_N,TAM081,0.4925045474502916,0.5968379446640316,0.8654398267990775,0.5984756097560976,0.8669086794587714,0.9980460269214068
TAM082_V1_UA_N,TAM082,0.8003310774623886,0.7728958101594364,0.9219702032312592,0.8261989694807769,0.9753399592464757,0.9445064583001116
TAM082_V2_UA_N,TAM082,0.560225558979739,0.5847899817375424,0.5424928491782615,0.8455919823339562,0.7135743160114828,0.7348104956268221
TAM082_V3_UA_N,TAM082,0.5747607655502392,0.6010632753049099,0.8062630781441662,0.6807556080283353,0.8773555276381909,0.908676913068228
TAM082_V5_UA_N,TAM082,0.5759632852089109,0.583987724523874,0.7316166710910539,0.7212931995540691,0.8506829230650513,0.8446866485013624
TAM082_V6_UA_N,TAM082,0.4045669646210471,0.376143585445911,0.9006562977613953,0.4237274801353158,0.9708305068320574,0.9257068933653668
TAM083_V1_UA_N,TAM083,0.2056704624680455,0.0904736562001064,0.8735065427650294,0.2229499937019775,0.9391922691220712,0.9324830448425954
TAM083_V2_UA_N,TAM083,0.1745329843876906,0.2843402464682897,0.8217866227742032,0.2942967322914552,0.8343530478132789,0.9844303484636296
TAM083_V3_UA_N,TAM083,0.6365621697780909,0.6935446993369407,0.8302591128678085,0.7493412384716732,0.8773130692035501,0.9395607794618002
TAM083_V4_UA_N,TAM083,0.6952968948610859,0.6263392433867451,0.9315443987592872,0.7071415390293412,0.987141444114738,0.943961423248338
TAM083_V5_UA_N,TAM083,0.3487657196087564,0.4287459735643674,0.8504293520686963,0.4304982722104559,0.852715604946001,0.9969219627014304
TAM083_V8_UA_N,TAM083,0.5435057975250901,0.3481677784885315,0.8099515255203878,0.5510204081632653,0.9917903140769752,0.8167944400659167
TAM083_V9_UA_N,TAM083,0.7188220230473752,0.677828245484314,0.7285451197053407,0.8481363996827914,0.8831342783792834,0.8209306706857573
TAM084_V2_UA_N,TAM084,0.8338808912251359,0.7560988031576267,0.8640966093822573,0.8647483690587139,0.9759219430310024,0.8876440315161452
TAM084_V3_UA_N,TAM084,0.8213990569247971,0.7921630094043887,0.9072267174361416,0.8554916090341926,0.970881616367418,0.9347104826605576
TAM084_V5_UA_N,TAM084,0.8556075231366305,0.7393254693470677,0.8008644877530902,0.900672974750797,0.9574361996283032,0.834363838857099
TAM084_V6_UA_N,TAM084,0.7477822883776876,0.6194597123143492,0.7998990408884402,0.7822202532580205,0.9750484570655016,0.8225319801511108
TAM084_V7_UA_N,TAM084,0.855854634212489,0.6448700211170174,0.7348455664075778,0.8809310653536258,0.98002357278854,0.753516409912927
TAM084_V9_UA_N,TAM084,0.8611138014527845,0.7215338993378559,0.8208113516463525,0.8811258114067687,0.9796520456418932,0.8350607047675451
TAM084_V10_UA_N,TAM084,0.7762299940723177,0.5673436230706742,0.7544798696765185,0.7851307801843663,0.990649636374748,0.7598912432027002
TAM085_V1_UA_N,TAM085,0.632492025295204,0.5369886444623937,0.8706453248352285,0.6395427795382526,0.9918726020050332,0.8769011926906664
TAM085_V3_UA_N,TAM085,0.5578231292517006,0.6900230997690023,0.8455932022611725,0.6906688687035508,0.8461045891141943,0.999285744296458
TAM085_V4_UA_N,TAM085,0.5516735564741966,0.5455342054202916,0.8223190619524686,0.6347680431141214,0.9146055064208264,0.8930068597560976
TAM085_V12_UA_N,TAM085,0.386735887686458,0.2149948552518145,0.4682369224894998,0.5189635497079949,0.9414163707897516,0.5116057971014493
TAM085_V13_LA_N,TAM085,0.2001966113456312,0.1688204027892914,0.8087250276661425,0.2216956967213114,0.9430502258294564,0.8502495330822694
TAM085_V13_UA_N,TAM085,0.5970817021709288,0.1957680717904566,0.4031768826106707,0.5999289664620224,0.9985483051462584,0.4041122973507315
TAM085_V15_LA_N,TAM085,0.2593025655134466,0.2701153543025005,0.915716250101108,0.2854762986521278,0.9444004171011472,0.9678963792587526
TAM085_V15_UA_N,TAM085,0.7036914746087302,0.3729246487867177,0.5039004613448903,0.7690474697435254,0.957293105279932,0.5367934589406328
TAM085_V16_LA_N,TAM085,0.2272438305496156,0.2130126225689365,0.8987119070059693,0.2416866433093613,0.965114709851552,0.9288868684244708
TAM085_V16_UA_N,TAM085,0.2410802388989872,0.4259030945629392,0.5189521104129007,0.6277735098739211,0.7401369416367786,0.7246123260437376
TAM085_V17_LA_N,TAM085,0.2693539272932755,0.3538108081110778,0.7570667759115117,0.4128205128205128,0.826264690853347,0.904910625290717
TAM085_V17_UA_N,TAM085,0.6184424501596097,0.533017679947615,0.8469506067867357,0.6450472416946053,0.9771693569963436,0.8657111699641689
TAM085_V18_UA_N,TAM085,0.7546721400021605,0.7151942080031308,0.8850250902368166,0.790870831984461,0.9679961556943776,0.9144163954613422
TAM085_V19_LA_N,TAM085,0.2222081859525012,0.0,0.5052018310445276,0.3933363148479427,0.6576200417536534,0.7872892347600519
TAM085_V19_UA_N,TAM085,0.2695279814933307,0.3928025802070956,0.3098156803950965,0.7906863223664942,0.571103434261329,0.5441845140032949
TAM085_V21_LA_N,TAM085,0.1618497109826589,0.0,0.4526582862414946,0.3190277250284846,0.5748018402527576,0.8117050977148175
TAM086_V3_UA_N,TAM086,0.4830139964669113,0.2309729194080283,0.5653663301856073,0.4934749409968069,0.985827844651374,0.5699990807737259
TAM086_V4_UA_N,TAM086,0.181518540317834,0.1421361303783905,0.6294701185579482,0.2602007897385604,0.9142535844813044,0.6768476962157495
TAM086_V5_UA_N,TAM086,0.0153537326388888,0.2552545411308468,0.7152281687941602,0.3361032829925091,0.8628749941131223,0.8386582253631724
TAM086_V6_UA_N,TAM086,0.09437894873966,0.1940285017022686,0.6854169204796631,0.26091888825865,0.804568113099616,0.8345614683180059
TAM086_V7_UA_N,TAM086,0.1926939437793272,0.1988370251374527,0.8826266740727026,0.2188955588213316,0.9290716559475086,0.9463973741330932
TAM086_V8_UA_N,TAM086,0.2008856579722016,0.177874481539403,0.8339527475444651,0.2220775913296924,0.9439887015835812,0.8773669217449589
TAM086_V9_UA_N,TAM086,0.1712284983728498,0.2154976961895706,0.8553938677978843,0.2195480878490397,0.8631885052696222,0.9895536758844174
TAM086_V10_UA_N,TAM086,0.2158142844436538,0.2232445164139555,0.935759496886017,0.2327794619900993,0.9574120851134604,0.9764019930032865
TAM088_V1_UA_N,TAM088,0.629935797910285,0.5115481061105979,0.813537859383544,0.6439903646482266,0.9913384998421992,0.8219123278276276
TAM088_V2_UA_N,TAM088,0.7421991737716446,0.6474732663352527,0.8322356908784321,0.7766425853887915,0.9761381826303838,0.8555659494855005
TAM088_V3_UA_N,TAM088,0.5234621037161453,0.7454853602945474,0.5134378748167563,0.9153386794062448,0.6091011095873332,0.8280374347790078
TAM088_V5_UA_N,TAM088,0.6657115020918104,0.5060200946607988,0.7378607694142502,0.7083163828674377,0.9637706961011216,0.7636488575846074
TAM088_V11_UA_N,TAM088,0.2234515797878407,0.5324892748770534,0.4868152625613902,0.6941754194516437,0.5852484331001907,0.8113399440630562
TAM088_V12_UA_N,TAM088,0.7083639276416794,0.6649622180557923,0.8552390640895219,0.7623318385650224,0.9627030392212348,0.892207600177436
TAM088_V13_UA_N,TAM088,0.3109653683116052,0.5947808607631286,0.539388733409072,0.7358557598365751,0.6330818264602576,0.841046397428648
TAM088_V15_UA_N,TAM088,0.3588390501319261,0.7712980381617844,0.5304413051631454,0.8467950136460869,0.5784200122691023,0.9203911452396664
TAM088_V16_UA_N,TAM088,0.6613897837091578,0.4918032786885246,0.7878259922687428,0.6659328563566318,0.9974495071344868,0.7902629863073245
TAM088_V18_UA_N,TAM088,0.6041760071735093,0.7483325311146256,0.7152797745701306,0.8323790585250455,0.7856794515046329,0.9257006632232035
TAM088_V19_UA_N,TAM088,0.4019551164135176,0.5882892173713762,0.6392400738057814,0.7264409018017168,0.7702886317783176,0.8416398969498425
TAM088_V20_UA_N,TAM088,0.4456909739928608,0.532093341701375,0.6613700025464732,0.6978971787772207,0.8393768987137225,0.8038809344385832
TAM100_V10_UA_N,TAM100,0.7523387061355351,0.3196054911368786,0.458444363411966,0.8353652589486004,0.9134442532032894,0.489517191805439
TAM100_V15_UA_N,TAM100,0.6183041076147009,0.3956469744080363,0.7606592077411551,0.6193082706766917,0.9987824675324676,0.7614363010329562
TAM100_V16_UA_N,TAM100,0.3056392887383573,0.3219004345236782,0.4562288444945908,0.6124213176668065,0.7093728231985759,0.6059040184706723
TAM100_V17_UA_N,TAM100,0.1623052488277113,0.368727804115478,0.4904159132007233,0.6173893251394105,0.7818108894013838,0.6721531418138563
"""

il = pd.read_csv(_io.StringIO(IL_CSV_TEXT))
assert len(il) == 185
print("inter-labeler:", il.shape, "| human-human avg:",
      round(il[['paul_vs_gbarimah','paul_vs_erik','gbarimah_vs_erik']].values.mean(), 4))

## Adapter — map the saved-model structures onto the analysis figures' variables

Everything below this cell is the tested `bruise_colab_final_analysis.ipynb` figure code,
which expects `CORE / DISP / PALETTE / per_image / per_image_custom / SUBJ / ITA / MAN /
SEG_MODELS / YOLO_BEST / cluster_ci / paired_delta / merged / fairness_analysis / BENCH_DF`.
This cell builds all of them from the saved-model run above. The **core YOLO path is native
argmax**; the custom /255 path is kept as `per_image_custom` for the two-path comparison.

In [ ]:
!pip install -q statsmodels

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.stats import spearmanr, kruskal, mannwhitneyu

# --- names / palette (fixed slot per model, colourblind-safe) ---
SEGFORMER_MODELS = ['segformer_b2_teacher', 'segformer_b0_direct', 'segformer_b0_distilled']
YOLO_MODELS      = ['yolo_sem_direct', 'yolo_sem_distilled']
CORE = SEGFORMER_MODELS + YOLO_MODELS
DISP = {r: MODELS[r]['display'] for r in CORE}
PALETTE = {'segformer_b2_teacher':'#2a78d6', 'segformer_b0_direct':'#1baf7a',
           'segformer_b0_distilled':'#eda100', 'yolo_sem_direct':'#008300', 'yolo_sem_distilled':'#4a3aa7'}
INK, MUTED, GRID = '#0b0b0b', '#898781', '#e1e0d9'
plt.rcParams.update({'figure.facecolor':'#fcfcfb','axes.facecolor':'#fcfcfb','axes.edgecolor':'#c3c2b7',
    'axes.labelcolor':INK,'text.color':INK,'xtick.color':MUTED,'ytick.color':MUTED,'grid.color':GRID,
    'axes.spines.top':False,'axes.spines.right':False,'font.size':10,'figure.dpi':120})

# --- per-image tables in the shape the figures want (enriched) ---
def enrich(df):
    df = df.copy()
    df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    tp = (df.recall * df.gt_positive_pixels).round()
    df['tp_pixels'] = tp
    df['fp_pixels'] = (df.pred_positive_pixels - tp).clip(lower=0)
    df['fn_pixels'] = (df.gt_positive_pixels - tp).clip(lower=0)
    df['pred_gt_ratio'] = df.pred_positive_pixels / df.gt_positive_pixels.replace(0, np.nan)
    return df

_saved_pi = dict(per_image)                     # demo's per_image: SegFormer + YOLO custom /255
per_image = {}
for s in SEGFORMER_MODELS: per_image[s] = enrich(_saved_pi[s])
for y in YOLO_MODELS:      per_image[y] = enrich(native_per_image[y])       # core YOLO = native argmax
per_image_custom = {y: enrich(_saved_pi[y]) for y in YOLO_MODELS}           # custom /255 (two-path fig)

# --- subject + ITA tables ---
SUBJ = ita[['stem', 'subject']].copy()
ITA  = ita[['stem', 'skin_tone_category', 'ita_group_index_5']].copy()
MAN  = {'test': test_df.merge(ITA, on='stem', how='left')}                  # image_path/mask_path are absolute

# --- handles for the gallery ---
SEG_MODELS = {s: (MODELS[s]['model'], MODELS[s]['cut']) for s in SEGFORMER_MODELS}
YOLO_BEST  = {y: MODELS[y]['ckpt'] for y in YOLO_MODELS}

# --- config + placeholders the spliced figures reference ---
CFG = {'drive_dir': DRIVE_DIR, 'img_size': IMG_H, 'render_gallery': True}
OUT_DIR = Path(DRIVE_DIR)                        # A1 tries OUT_DIR/FINAL_RESULTS.csv (absent -> handled)
BEST = {r: {'seed': 'saved', 'cut': MODELS[r].get('cut')} for r in CORE}

# --- benchmark table for the speed figure (from the demo's `bench` dict) ---
BENCH_DF = pd.DataFrame([{'model': run, 'median_ms': bench[run]['median_ms'],
                          'p95_ms': bench[run]['p95_ms'], 'fps': bench[run]['fps'],
                          'params_M': MODELS[run]['params_m']} for run in CORE])

# --- subject-level bootstrap helpers (verbatim from the final-analysis notebook) ---
RNG_SEED = 42
def _subject_index(df):
    subs = df['subject'].unique(); return subs, {s: df[df.subject == s] for s in subs}
def cluster_ci(frame, stat, B=4000, seed=RNG_SEED):
    rng = np.random.default_rng(seed); subs, by = _subject_index(frame)
    vals = np.array([stat(pd.concat([by[s] for s in rng.choice(subs, len(subs), True)], ignore_index=True)) for _ in range(B)])
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))
def paired_delta(frame, stat, B=4000, seed=RNG_SEED):
    rng = np.random.default_rng(seed); subs, by = _subject_index(frame); point = stat(frame)
    vals = np.array([stat(pd.concat([by[s] for s in rng.choice(subs, len(subs), True)], ignore_index=True)) for _ in range(B)])
    return {'delta': float(point), 'ci_lo': float(np.percentile(vals, 2.5)),
            'ci_hi': float(np.percentile(vals, 97.5)), 'p_gt0': float((vals > 0).mean())}
def merged(a, b, cols=('dice',)):
    left = per_image[a][['stem', *cols]].rename(columns={c: f'{c}_a' for c in cols})
    right = per_image[b][['stem', *cols]].rename(columns={c: f'{c}_b' for c in cols})
    return left.merge(right, on='stem').merge(SUBJ, on='stem')

# --- fairness_analysis (ported from bruisekit.evaluate; same output schema) ---
from scipy import stats as _st
def _bootstrap_ci(values, n=2000, seed=0):
    if len(values) < 2: return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    meds = [np.median(rng.choice(values, size=len(values), replace=True)) for _ in range(n)]
    return float(np.percentile(meds, 2.5)), float(np.percentile(meds, 97.5))
def fairness_analysis(per_image_df, manifest, model_name):
    df = per_image_df.merge(manifest[['stem', 'skin_tone_category', 'ita_group_index_5']],
                            on='stem', how='left', validate='one_to_one')
    per_group, samples = [], []
    for gidx, g in sorted(df.groupby('ita_group_index_5'), key=lambda kv: kv[0]):
        vals = g['dice'].to_numpy(); lo, hi = _bootstrap_ci(vals)
        per_group.append({'model': model_name, 'ita_group_index_5': int(gidx),
                          'skin_tone_category': g['skin_tone_category'].iloc[0], 'n_images': len(g),
                          'median_dice': float(np.median(vals)),
                          'iqr_dice': float(np.percentile(vals, 75) - np.percentile(vals, 25)),
                          'ci95_lo': lo, 'ci95_hi': hi, 'mean_recall': float(g['recall'].mean()),
                          'miss_rate': float(((g['pred_positive_pixels'] == 0) & (g['gt_positive_pixels'] > 0)).mean())})
        samples.append(vals)
    H, p = _st.kruskal(*samples)
    pairs = [(i, j) for i in range(len(samples)) for j in range(i + 1, len(samples))]
    pairwise = []
    for i, j in pairs:
        pv = _st.mannwhitneyu(samples[i], samples[j], alternative='two-sided').pvalue
        adj = min(1.0, pv * len(pairs))
        pairwise.append({'model': model_name, 'group_a': per_group[i]['skin_tone_category'],
                         'group_b': per_group[j]['skin_tone_category'], 'pvalue': pv,
                         'bonferroni_p': adj, 'significant': bool(adj < 0.05)})
    pg = pd.DataFrame(per_group)
    best, worst = pg.loc[pg['median_dice'].idxmax()], pg.loc[pg['median_dice'].idxmin()]
    stat = {'model': model_name, 'kruskal_H': float(H), 'kruskal_p': float(p), 'significant': bool(p < 0.05),
            'fairness_gap': float(best['median_dice'] - worst['median_dice']),
            'best_group': best['skin_tone_category'], 'worst_group': worst['skin_tone_category'],
            'max_miss_rate_gap': float(pg['miss_rate'].max() - pg['miss_rate'].min())}
    return {'per_group': pg, 'pairwise': pd.DataFrame(pairwise), 'stats': stat}

print("adapter ready:", len(CORE), "core variants |", {k: len(v) for k, v in per_image.items()})

---
## Qualitative galleries (saved-model inference path)

These use inference_demo's `bruise_mask_640` for SegFormer/YOLO-custom (correct ImageNet
normalisation for the pipeline SegFormer) and native `.predict()` for the core YOLO number.
Three views: easy→hard, by ITA skin-tone group, and by bruise-size quartile.

In [ ]:
def _rgb640(p):
    im = cv2.imread(str(p)); return cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), (IMG_W, IMG_H)) if im is not None else None
def _overlay(img, mask, color=(230, 60, 60), a=0.45):
    lay = np.zeros_like(img); lay[mask.astype(bool)] = color; return cv2.addWeighted(lay, a, img, 1 - a, 0)
_STEM2IDX = {s: i for i, s in enumerate(STEMS)}
def saved_predict_mask(run, stem, img_path):
    if run in SEG_MODELS:                                   # SegFormer: correct via staged /ImageNet tensor
        idx = _STEM2IDX[stem]
        return bruise_mask_640(MODELS[run], X_TEST[idx:idx+1])[0].to(torch.uint8).cpu().numpy()
    res = _YOLO_native(YOLO_BEST[run]).predict(str(img_path), imgsz=IMG_H, device=0, verbose=False)[0]
    cm = res.semantic_mask.data if getattr(res, 'semantic_mask', None) is not None else np.zeros((IMG_H, IMG_W))
    cm = cm.cpu().numpy() if hasattr(cm, 'cpu') else np.asarray(cm)
    return _to640_nn((cm == 1).astype('uint8'))

def gallery(picks, row_labels, title):
    tdf = MAN['test'].set_index('stem')
    fig, axes = plt.subplots(len(picks), len(CORE)+1, figsize=(2.7*(len(CORE)+1), 2.7*len(picks)))
    axes = np.atleast_2d(axes)
    for i, (stem, lab) in enumerate(zip(picks, row_labels)):
        row = tdf.loc[stem]; img = _rgb640(row.image_path)
        gt = (GT_640[_STEM2IDX[stem], 0].numpy() > 0.5).astype('uint8')
        axes[i, 0].imshow(_overlay(img, gt, (40, 190, 40))); axes[i, 0].set_ylabel(lab, fontsize=7)
        if i == 0: axes[i, 0].set_title('Ground truth', fontsize=9)
        for j, run in enumerate(CORE, start=1):
            m = saved_predict_mask(run, stem, row.image_path)
            axes[i, j].imshow(_overlay(img, m))
            axes[i, j].set_xlabel(f"Dice {per_image[run].set_index('stem').dice[stem]:.2f}", fontsize=7)
            if i == 0: axes[i, j].set_title(DISP[run], fontsize=7)
    for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
    fig.suptitle(title, fontsize=12, y=1.01); plt.tight_layout(); plt.show()

# easy -> hard (by mean Dice across core models)
_D = pd.DataFrame({r: per_image[r].set_index('stem').dice for r in CORE})
_ord = _D.mean(axis=1).sort_values()
_picks = [_ord.index[k] for k in [len(_ord)-1, int(len(_ord)*0.6), int(len(_ord)*0.25), 0]]
gallery(_picks, [f'{lab}\n{s}' for lab, s in zip(['easiest','typical','hard','hardest'], _picks)],
        'Predictions across easy → hard (green = truth, red = model)')

In [ ]:
# by ITA skin-tone group (typical image per group = closest to that group's median Dice)
_ref = 'segformer_b0_distilled'; _dref = per_image[_ref].set_index('stem').dice
_ORDER = ["Light (II-III)","Intermediate (III-IV)","Tan (IV)","Brown (V)","Dark (VI)"]
_groups = [g for g in _ORDER if g in ITA.skin_tone_category.unique()]
_picks = []
for g in _groups:
    st = ITA[ITA.skin_tone_category == g].stem; dd = _dref.loc[_dref.index.isin(st)]
    _picks.append(dd.sub(dd.median()).abs().idxmin())
gallery(_picks, [f"{g.split(' ')[0]}\n{s}" for g, s in zip(_groups, _picks)],
        'Typical prediction per ITA skin-tone group')

In [ ]:
# by bruise-size quartile (typical image per quartile = closest to that quartile's median size)
_sz = per_image[_ref][['stem','gt_positive_pixels']].copy()
_sz['q'] = pd.qcut(_sz.gt_positive_pixels, 4, labels=['Q1 smallest','Q2','Q3','Q4 largest'])
_picks, _labs = [], []
for b in ['Q1 smallest','Q2','Q3','Q4 largest']:
    s = _sz[_sz.q == b].set_index('stem').gt_positive_pixels
    stem = s.sub(s.median()).abs().idxmin(); _picks.append(stem)
    _labs.append(f"{b}\n{int(s[stem]):,} px")
gallery(_picks, _labs, 'Typical prediction per bruise-size quartile')

---
# A · Accuracy & distribution

## A1 · Headline table (best seed) + on-disk 3-seed spread

Left: this best-seed run. Right (printed): the 3-seed mean±std already on disk in
`results_final/FINAL_RESULTS.csv`, so you can see the best seed sitting inside the
seed spread rather than cherry-picked outside it.

In [ ]:
rows = []
for name in CORE:
    d = per_image[name]
    rows.append({"variant": DISP[name], "median_dice": d.dice.median(), "mean_dice": d.dice.mean(),
                 "mean_iou": d.iou.mean(), "miss_%": d.complete_miss.mean()*100,
                 "seed": BEST[name]["seed"]})
head = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(head))[::-1]
ax.barh(y, head.mean_dice, height=0.6, color=[PALETTE[n] for n in CORE], zorder=3)
for yy, m, md_ in zip(y, head.mean_dice, head.median_dice):
    ax.text(m+0.006, yy, f"mean {m:.3f} / med {md_:.3f}", va="center", fontsize=8.5, color=INK)
ax.set_yticks(y); ax.set_yticklabels(head.variant, fontsize=9)
ax.set_xlim(0, 1.02); ax.set_xlabel("Dice (best seed, 185 test images)")
ax.set_title("Headline accuracy — best val-selected seed", fontsize=12, pad=10)
ax.grid(axis="x", lw=0.6, zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

try:
    disk = pd.read_csv(OUT_DIR/"FINAL_RESULTS.csv")
    print("On-disk 3-seed mean±std (results_final/FINAL_RESULTS.csv):")
    print(disk[["variant","dice","median_dice","miss_%"]].to_string(index=False))
except Exception as e:
    print("FINAL_RESULTS.csv not read:", e)
head.round(4)

## A2 · Per-image Dice: violins + survival curves

The violin shows the spread over 185 images (white dot = median, red dash = mean).
The survival curve on the right, P(Dice ≥ x), makes the **worst-case tail** explicit:
where a curve drops early, that model has more low-Dice images — the failures a single
average number hides.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, 4.6))
data = [per_image[n].dice.values for n in CORE]
parts = a1.violinplot(data, positions=np.arange(len(CORE)), widths=0.75,
                      showmeans=False, showmedians=False, showextrema=False)
for pc, n in zip(parts["bodies"], CORE):
    pc.set_facecolor(PALETTE[n]); pc.set_alpha(0.55); pc.set_edgecolor("#fcfcfb"); pc.set_linewidth(2)
for i, v in enumerate(data):
    q1, med, q3 = np.percentile(v, [25, 50, 75])
    a1.vlines(i, q1, q3, color=INK, lw=5, zorder=3)
    a1.plot(i, med, "o", color="#fcfcfb", ms=6, zorder=4, markeredgecolor=INK, markeredgewidth=1.2)
    a1.plot(i, v.mean(), "_", color="#e34948", ms=16, mew=2.5, zorder=5)
a1.set_xticks(range(len(CORE))); a1.set_xticklabels([DISP[n] for n in CORE], rotation=20, ha="right", fontsize=7.5)
a1.set_ylabel("Per-image Dice"); a1.set_ylim(-0.02, 1.02)
a1.set_title("Distribution (white=median, red=mean)", fontsize=11); a1.grid(axis="y", lw=0.6); a1.set_axisbelow(True)

xs = np.linspace(0, 1, 200)
for n in CORE:
    v = per_image[n].dice.values
    surv = [(v >= x).mean() for x in xs]
    a2.plot(xs, surv, color=PALETTE[n], lw=2, label=DISP[n])
a2.set_xlabel("Dice threshold x"); a2.set_ylabel("P(Dice ≥ x)")
a2.set_title("Survival curves — the low-Dice tail", fontsize=11)
a2.grid(lw=0.6); a2.set_axisbelow(True)
a2.legend(fontsize=7, loc="lower left", frameon=False)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "median": per_image[n].dice.median(), "mean": per_image[n].dice.mean(),
               "p25": per_image[n].dice.quantile(.25), "p05": per_image[n].dice.quantile(.05),
               "zeros": int((per_image[n].dice==0).sum())} for n in CORE]).round(4)

## A3 · Precision vs recall, and IoU vs Dice

Left: each dot is one image, precision vs recall. It shows an operating-point
character no single number does — e.g. YOLO tends to sit high-precision / lower-recall
(it fires *cleanly* but *misses*), which is exactly the under-detection the miss-rate
and fairness sections pick up. Right: IoU vs Dice, a consistency check (they should
track monotonically; the scatter is the per-image agreement).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13.5, 4.8))
for n in CORE:
    d = per_image[n]
    a1.scatter(d.recall, d.precision, s=14, alpha=0.45, color=PALETTE[n], edgecolors="none", label=DISP[n])
    a1.plot(d.recall.mean(), d.precision.mean(), "o", ms=11, color=PALETTE[n], markeredgecolor=INK, mew=1.2, zorder=5)
a1.set_xlabel("recall  (1 − under-segmentation)"); a1.set_ylabel("precision  (1 − over-segmentation)")
a1.set_xlim(0,1.02); a1.set_ylim(0,1.02); a1.set_title("Per-image precision vs recall (big dot = mean)", fontsize=11)
a1.grid(lw=0.6); a1.set_axisbelow(True); a1.legend(fontsize=7, loc="lower left", frameon=False)

for n in CORE:
    d = per_image[n]
    a2.scatter(d.dice, d.iou, s=13, alpha=0.4, color=PALETTE[n], edgecolors="none")
a2.plot([0,1],[0,1], color=MUTED, ls="--", lw=1)
a2.set_xlabel("Dice"); a2.set_ylabel("IoU"); a2.set_xlim(0,1.02); a2.set_ylim(0,1.02)
a2.set_title("IoU vs Dice (dashed = y=x)", fontsize=11); a2.grid(lw=0.6); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "mean_precision": per_image[n].precision.mean(),
               "mean_recall": per_image[n].recall.mean(), "mean_iou": per_image[n].iou.mean()}
              for n in CORE]).round(4)

---
# B · Safety & failure modes

## B1 · Complete-miss rate — the failure that matters, and "best Dice ≠ safest"

A **complete miss** = a real bruise, empty predicted mask. For an injury-documentation
tool that is the decisive failure: a loose outline is correctable, a blank mask is
silence about an injury that is present. The right panel plots miss vs median Dice —
watch the model with the best Dice not being the safest.

In [ ]:
rows = []
for n in CORE:
    f = per_image[n][["stem","complete_miss"]].merge(SUBJ, on="stem")
    f["complete_miss"] = f.complete_miss.astype(float)
    lo, hi = cluster_ci(f, lambda x: x.complete_miss.mean()*100, B=3000)
    rows.append({"model": n, "miss_pct": f.complete_miss.mean()*100, "lo": lo, "hi": hi,
                 "median_dice": per_image[n].dice.median()})
miss = pd.DataFrame(rows)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
xs = np.arange(len(miss))
a1.bar(xs, miss.miss_pct, width=0.55, color=[PALETTE[n] for n in miss.model], zorder=3)
a1.errorbar(xs, miss.miss_pct, yerr=[miss.miss_pct-miss.lo, miss.hi-miss.miss_pct],
            fmt="none", ecolor=INK, elinewidth=1.5, capsize=5, zorder=4)
for x, v, hi in zip(xs, miss.miss_pct, miss.hi):
    a1.text(x, hi+0.3, f"{v:.1f}%", ha="center", fontsize=9, color=INK)
a1.set_xticks(xs); a1.set_xticklabels([DISP[n] for n in miss.model], rotation=20, ha="right", fontsize=7.5)
a1.set_ylabel("Complete-miss rate (%)"); a1.set_title("Blank-mask failures (lower better)", fontsize=11)
a1.grid(axis="y", lw=0.6, zorder=0); a1.set_axisbelow(True)
for _, r in miss.iterrows():
    a2.scatter(r.miss_pct, r.median_dice, s=150, color=PALETTE[r.model], edgecolors=INK, lw=0.6, zorder=3)
    a2.annotate(DISP[r.model], (r.miss_pct, r.median_dice), fontsize=7.5, textcoords="offset points", xytext=(7,5))
a2.set_xlabel("Complete-miss rate (%)  →  worse"); a2.set_ylabel("Median Dice  →  better")
a2.set_title("Best Dice ≠ safest model", fontsize=11); a2.grid(lw=0.6, zorder=0); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
miss.assign(model=lambda d: d.model.map(DISP)).round(3)

## B2 · Over- vs under-segmentation — *how* they fail

`pred / GT area` = predicted bruise area ÷ true area. **<1** under-segments (misses
bruise — bad for evidence); **>1** over-segments (flags healthy skin — recoverable).
Which way a model leans matters more forensically than its total error.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.4))
rng = np.random.default_rng(0)
for i, n in enumerate(CORE):
    v = per_image[n].pred_gt_ratio.replace([np.inf,-np.inf], np.nan).dropna().clip(0, 3)
    ax.scatter(rng.normal(i, 0.07, len(v)), v, s=13, alpha=0.5, color=PALETTE[n], edgecolors="none", zorder=3)
    ax.plot(i, v.median(), "_", color=INK, ms=28, mew=2.5, zorder=4)
ax.axhline(1.0, color="#e34948", ls="--", lw=1.4, zorder=2)
ax.text(len(CORE)-0.45, 1.04, "perfect size", color="#e34948", fontsize=8.5)
ax.set_xticks(range(len(CORE))); ax.set_xticklabels([DISP[n] for n in CORE], rotation=20, ha="right", fontsize=7.5)
ax.set_ylabel("pred / GT area (clipped at 3)")
ax.set_title("Under- (<1) vs over- (>1) segmentation; black dash = median", fontsize=11.5, pad=10)
ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n], "median_pred_gt_ratio": per_image[n].pred_gt_ratio.median(),
               "under_seg_%": (per_image[n].pred_gt_ratio < 1).mean()*100,
               "mean_FP_px": per_image[n].fp_pixels.mean(), "mean_FN_px": per_image[n].fn_pixels.mean()}
              for n in CORE]).round(2)

## B3 · YOLO: native argmax vs custom /255 — which path, and where they differ

YOLO is evaluated two ways: **native argmax** (its home turf, letterbox + argmax) and
**custom /255** (the same 640 geometry SegFormer sees, threshold swept on val). Same
weights, different eval. The paired per-image delta shows the paths are not
interchangeable — report both and say which is which.

In [ ]:
fig, axes = plt.subplots(1, len(YOLO_MODELS), figsize=(11, 4.4))
comp_rows = []
for ax, n in zip(np.atleast_1d(axes), YOLO_MODELS):
    m = per_image[n][["stem","dice"]].rename(columns={"dice":"native"}).merge(
        per_image_custom[n][["stem","dice"]].rename(columns={"dice":"custom"}), on="stem")
    ax.scatter(m.native, m.custom, s=16, alpha=0.5, color=PALETTE[n], edgecolors="none")
    ax.plot([0,1],[0,1], color=MUTED, ls="--", lw=1)
    ax.set_xlabel("native argmax Dice"); ax.set_ylabel("custom /255 Dice")
    ax.set_xlim(0,1.02); ax.set_ylim(0,1.02); ax.set_title(DISP[n], fontsize=10)
    ax.grid(lw=0.6); ax.set_axisbelow(True)
    comp_rows.append({"model": DISP[n], "native_median": per_image[n].dice.median(),
                      "custom_median": per_image_custom[n].dice.median(),
                      "native_miss_%": per_image[n].complete_miss.mean()*100,
                      "custom_miss_%": per_image_custom[n].complete_miss.mean()*100,
                      "mean_abs_delta": (m.native-m.custom).abs().mean()})
fig.suptitle("YOLO evaluation paths are not interchangeable (dashed = agreement)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
pd.DataFrame(comp_rows).round(4)

---
# C · Inference statistics (which differences are real?)

The 185 images come from only **28 people**, so photos of one person are not
independent. Every interval below is a **subject-level cluster bootstrap** (resample
people, B=4000); model-vs-model contrasts are **paired** (the same resample scores
both models — correct for a shared test set and ~2× tighter than unpaired).

## C1 · Marginal CIs — mean Dice per model

In [ ]:
rows = []
for n in CORE:
    f = per_image[n][["stem","dice"]].merge(SUBJ, on="stem")
    lo, hi = cluster_ci(f, lambda x: x.dice.mean(), B=4000)
    rows.append({"model": n, "mean": f.dice.mean(), "lo": lo, "hi": hi})
ci = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.5, 4))
y = np.arange(len(ci))[::-1]
for yy, (_, r) in zip(y, ci.iterrows()):
    ax.plot([r.lo, r.hi], [yy, yy], color=PALETTE[r.model], lw=3, solid_capstyle="round", zorder=3)
    ax.plot(r["mean"], yy, "o", ms=9, color=PALETTE[r.model], markeredgecolor="#fcfcfb", mew=1.5, zorder=4)
    ax.text(r.hi+0.004, yy, f"{r['mean']:.3f} [{r.lo:.3f}, {r.hi:.3f}]", va="center", fontsize=8.5)
ax.set_yticks(y); ax.set_yticklabels([DISP[n] for n in ci.model], fontsize=9)
ax.set_xlabel("Mean Dice (95% subject cluster-bootstrap)"); ax.set_xlim(0.55, 0.92)
ax.set_title("Overlapping intervals: at 28 subjects the models are hard to separate", fontsize=11, pad=10)
ax.grid(axis="x", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
ci.assign(model=lambda d: d.model.map(DISP)).round(4)

## C2 · Paired contrasts — forest plot with P(Δ>0)

Each row is one paired comparison: Δ mean Dice with its 95% CI and the one-sided
**P(Δ>0)** (a 92% and a 52% are both "n.s." but mean very different things). A
contrast is resolvable only if its interval clears zero.

In [ ]:
CONTRASTS = [
    ("Distillation (B0-dist − B0-direct)", "segformer_b0_distilled", "segformer_b0_direct", "dice"),
    ("Student − teacher (B0-dist − B2)",   "segformer_b0_distilled", "segformer_b2_teacher", "dice"),
    ("SegFormer − YOLO (B0-dist − YOLO-d)","segformer_b0_distilled", "yolo_sem_direct",      "dice"),
    ("YOLO distill − direct",              "yolo_sem_distilled",     "yolo_sem_direct",      "dice"),
]
res = []
for label, a, b, col in CONTRASTS:
    m = merged(a, b, cols=(col,))
    r = paired_delta(m, lambda x: x[f"{col}_a"].mean() - x[f"{col}_b"].mean(), B=4000)
    r["label"] = label; res.append(r)
# add the miss-rate contrast that historically IS significant
mm = per_image["yolo_sem_direct"][["stem","complete_miss"]].rename(columns={"complete_miss":"a"}).merge(
     per_image["segformer_b0_distilled"][["stem","complete_miss"]].rename(columns={"complete_miss":"b"}), on="stem").merge(SUBJ,on="stem")
mm["a"]=mm.a.astype(float); mm["b"]=mm.b.astype(float)
rm = paired_delta(mm, lambda x: (x.a.mean()-x.b.mean())*100, B=4000); rm["label"]="MISS %: YOLO-direct − B0-dist"
FOREST = pd.DataFrame(res + [rm])

fig, ax = plt.subplots(figsize=(9.5, 4.4))
y = np.arange(len(FOREST))[::-1]
for yy, (_, r) in zip(y, FOREST.iterrows()):
    sig = r.ci_lo > 0 or r.ci_hi < 0
    c = "#c0392b" if sig else MUTED
    ax.plot([r.ci_lo, r.ci_hi], [yy, yy], color=c, lw=3, solid_capstyle="round", zorder=3)
    ax.plot(r.delta, yy, "o", ms=8, color=c, zorder=4)
    ax.text(r.ci_hi+0.002 if r.ci_hi>=0 else r.ci_hi, yy+0.28,
            f"Δ={r.delta:+.3f}  P(Δ>0)={r.p_gt0*100:.0f}%", fontsize=8, color=INK)
ax.axvline(0, color=INK, lw=1.1, zorder=2)
ax.set_yticks(y); ax.set_yticklabels(FOREST.label, fontsize=8.5)
ax.set_xlabel("Δ (paired subject bootstrap; Dice contrasts, except the miss-% row in %)")
ax.set_title("Paired effect sizes — red = interval clears zero", fontsize=11, pad=10)
ax.grid(axis="x", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
FOREST[["label","delta","ci_lo","ci_hi","p_gt0"]].round(4)

## C3 · Per-subject heatmap + model agreement

Left: each of the 28 subjects × each model, mean Dice (darker = better; hardest
subjects on top). Rows dark across *every* column are hard **subjects**, not model
failures — and with only 28 rows, one or two of them move the whole average (why C1's
bars are wide). Right: Spearman correlation of per-image Dice between models — high
everywhere means the models agree on which images are hard, so an ensemble helps less
than you'd hope.

In [ ]:
mats = []
for n in CORE:
    s = per_image[n][["stem","dice"]].merge(SUBJ, on="stem").groupby("subject").dice.mean()
    mats.append(s.rename(DISP[n]))
H = pd.concat(mats, axis=1)
H = H.loc[H.mean(axis=1).sort_values().index]
D = pd.DataFrame({DISP[n]: per_image[n].set_index("stem").dice for n in CORE})
C = D.corr(method="spearman")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 8), gridspec_kw={"width_ratios":[1, 1.05]})
im1 = a1.imshow(H.values, cmap="Blues", vmin=0, vmax=1, aspect="auto")
a1.set_xticks(range(len(H.columns))); a1.set_xticklabels(H.columns, rotation=35, ha="right", fontsize=7.5)
a1.set_yticks(range(len(H))); a1.set_yticklabels(H.index, fontsize=7)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        v = H.values[i, j]
        a1.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6,
                color="#ffffff" if v > 0.55 else INK)
a1.set_title("Per-subject mean Dice (hardest on top)", fontsize=11)
fig.colorbar(im1, ax=a1, shrink=0.5, label="Mean Dice")

im2 = a2.imshow(C.values, cmap="Blues", vmin=0, vmax=1)
a2.set_xticks(range(len(C))); a2.set_xticklabels(C.columns, rotation=35, ha="right", fontsize=7.5)
a2.set_yticks(range(len(C))); a2.set_yticklabels(C.index, fontsize=7.5)
for i in range(len(C)):
    for j in range(len(C)):
        a2.text(j, i, f"{C.values[i,j]:.2f}", ha="center", va="center", fontsize=8,
                color="#ffffff" if C.values[i,j] > 0.55 else INK)
a2.set_title("Spearman corr. of per-image Dice", fontsize=11)
fig.colorbar(im2, ax=a2, shrink=0.7, label="ρ")
plt.tight_layout(); plt.show()
print("Hardest 5 subjects (mean over models):")
print(H.mean(axis=1).head(5).round(3).to_string())

---
# D · ⭐ Fairness across skin tone (ITA)

This is a forensic tool: a model that documents bruises well on light skin and poorly
on dark skin has an **evidentiary** problem, not a metric one. Groups are ITA
(Individual Typology Angle) — an objective, pixel-computed skin-tone measure. Computed
here on the **best-seed** per-image data via the same `fairness_analysis` used to make
`results_final/fairness_*.csv` (which are 3-seed-averaged).

**Honest caveat up front:** each ITA group holds only ~9–17 *subjects*. The
group-level CIs are wide and mostly overlap — treat these as **exploratory**. The
gap's *sign* is a hypothesis, not a proven effect, at n=28.

In [ ]:
# Fairness fresh from best-seed per-image, for the 5 core variants + both YOLO paths.
fair_frames = {}
for n in SEGFORMER_MODELS:
    fair_frames[DISP[n]] = per_image[n]
for n in YOLO_MODELS:
    fair_frames[f"{DISP[n]} · native"] = per_image[n]
    fair_frames[f"{DISP[n]} · custom"] = per_image_custom[n]

fg, fp, fs = [], [], []
for label, pi in fair_frames.items():
    out = fairness_analysis(pi[["stem","dice","recall","pred_positive_pixels","gt_positive_pixels"]],
                            MAN["test"], label)
    fg.append(out["per_group"]); fp.append(out["pairwise"]); fs.append(out["stats"])
FAIR_GROUP = pd.concat(fg, ignore_index=True)
FAIR_PAIR  = pd.concat(fp, ignore_index=True)
FAIR_STATS = pd.DataFrame(fs)
print(FAIR_STATS[["model","kruskal_p","significant","fairness_gap","best_group","worst_group","max_miss_rate_gap"]].to_string(index=False))

## D1 · Per-group heatmaps — median Dice, recall, miss-rate

In [ ]:
GROUP_ORDER = ["Light (II-III)","Intermediate (III-IV)","Tan (IV)","Brown (V)","Dark (VI)"]
def heat(ax, values, title, cmap, fmt, vmax):
    piv = FAIR_GROUP.pivot_table(index="model", columns="skin_tone_category", values=values)
    piv = piv.reindex(columns=[g for g in GROUP_ORDER if g in piv.columns])
    im = ax.imshow(piv.values, cmap=cmap, aspect="auto", vmin=0, vmax=vmax)
    ax.set_xticks(range(piv.shape[1])); ax.set_xticklabels(piv.columns, rotation=30, ha="right", fontsize=7)
    ax.set_yticks(range(piv.shape[0])); ax.set_yticklabels(piv.index, fontsize=7)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, fmt(piv.values[i,j]), ha="center", va="center", fontsize=6.5,
                    color="#ffffff" if (piv.values[i,j]/vmax) > 0.55 else INK)
    ax.set_title(title, fontsize=10)
    return im

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
i1 = heat(axes[0], "median_dice", "Median Dice", "Blues", lambda v: f"{v:.2f}", 1.0)
i2 = heat(axes[1], "mean_recall", "Mean recall", "Greens", lambda v: f"{v:.2f}", 1.0)
fair_miss = FAIR_GROUP.assign(miss_pct=FAIR_GROUP.miss_rate*100)
piv = fair_miss.pivot_table(index="model", columns="skin_tone_category", values="miss_pct").reindex(columns=[g for g in GROUP_ORDER if g in fair_miss.skin_tone_category.unique()])
im3 = axes[2].imshow(piv.values, cmap="Reds", aspect="auto", vmin=0, vmax=max(1e-6, np.nanmax(piv.values)))
axes[2].set_xticks(range(piv.shape[1])); axes[2].set_xticklabels(piv.columns, rotation=30, ha="right", fontsize=7)
axes[2].set_yticks(range(piv.shape[0])); axes[2].set_yticklabels(piv.index, fontsize=7)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        axes[2].text(j, i, f"{piv.values[i,j]:.0f}", ha="center", va="center", fontsize=6.5, color=INK)
axes[2].set_title("Complete-miss rate (%)", fontsize=10)
for im, ax in [(i1,axes[0]),(i2,axes[1]),(im3,axes[2])]:
    fig.colorbar(im, ax=ax, shrink=0.6)
fig.suptitle("Performance by ITA skin-tone group (rows = variants)", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()
FAIR_GROUP.pivot_table(index="model", columns="skin_tone_category", values="median_dice").round(3)

## D2 · Fairness gap + the light-skin under-detection story

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6))
# gap bars, coloured by omnibus significance
order = FAIR_STATS.sort_values("fairness_gap")
xs = np.arange(len(order))
cols = ["#c0392b" if s else MUTED for s in order.significant]
a1.barh(xs, order.fairness_gap, color=cols, zorder=3)
for x, g, p in zip(xs, order.fairness_gap, order.kruskal_p):
    a1.text(g+0.003, x, f"{g:.3f} (KW p={p:.2f})", va="center", fontsize=7.5)
a1.set_yticks(xs); a1.set_yticklabels(order.model, fontsize=7.5)
a1.set_xlabel("fairness gap = best − worst group median Dice")
a1.set_title("Max−min Dice gap (red = Kruskal p<0.05)", fontsize=10.5)
a1.grid(axis="x", lw=0.6); a1.set_axisbelow(True)

# precision vs recall per group for one YOLO path -> under-detection vs confusion
yl = f"{DISP['yolo_sem_direct']} · native"
sub = FAIR_GROUP[FAIR_GROUP.model == yl]
# recompute per-group precision on the fly for this panel
pi = per_image["yolo_sem_direct"].merge(ITA, on="stem")
grp = pi.groupby("skin_tone_category").agg(recall=("recall","mean"), precision=("precision","mean"),
                                           miss=("complete_miss","mean")).reindex([g for g in GROUP_ORDER if g in pi.skin_tone_category.unique()])
a2.scatter(grp.recall, grp.precision, s=140, c=range(len(grp)), cmap="viridis", edgecolors=INK, zorder=3)
for g, r in grp.iterrows():
    a2.annotate(g.split(" ")[0], (r.recall, r.precision), fontsize=7.5, textcoords="offset points", xytext=(6,4))
a2.set_xlabel("recall"); a2.set_ylabel("precision")
a2.set_title(f"{DISP['yolo_sem_direct']}: per-group precision vs recall\n(low recall + high precision = under-detection)", fontsize=9.5)
a2.grid(lw=0.6); a2.set_axisbelow(True)
plt.tight_layout(); plt.show()
grp.assign(miss_pct=lambda d: d.miss*100).round(3)

## D3 · Pairwise significance (Bonferroni) — mostly n.s. at n=28

In [ ]:
sig = FAIR_PAIR.groupby("model").significant.sum()
tot = FAIR_PAIR.groupby("model").significant.count()
tab = pd.DataFrame({"significant_pairs": sig, "of_pairs": tot}).reset_index()
print("Bonferroni-significant group pairs per variant (out of 10):")
print(tab.to_string(index=False))
print("\nInterpretation: with ~9-17 subjects per ITA group, almost nothing survives")
print("Bonferroni. The heatmaps show DIRECTION; do not claim a proven per-group gap at n=28.")
FAIR_PAIR[FAIR_PAIR.significant][["model","group_a","group_b","bonferroni_p"]].round(4)

---
# E · ⭐ Bruise size

Small bruises are intrinsically harder — a few pixels of error costs proportionally
far more Dice on a small target. So a model's score partly reflects *which bruises*
were in the test set, and — crucially for section D — **if bruise size correlates with
skin tone, an apparent fairness gap can be a size effect in disguise.** This section
tests exactly that.

## E1 · Size distribution + Dice-vs-size + miss-vs-size

In [ ]:
sizes = per_image[CORE[0]].set_index("stem").gt_positive_pixels
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
axes[0].hist(np.log10(sizes.clip(lower=1)), bins=30, color="#2a78d6", alpha=0.85)
axes[0].set_xlabel("log10(bruise size, px)"); axes[0].set_ylabel("images")
axes[0].set_title(f"Bruise-size distribution (median {int(sizes.median()):,} px)", fontsize=10)
axes[0].grid(lw=0.5); axes[0].set_axisbelow(True)

for n in CORE:
    d = per_image[n]
    axes[1].scatter(d.gt_positive_pixels, d.dice, s=12, alpha=0.35, color=PALETTE[n], edgecolors="none")
axes[1].set_xscale("log"); axes[1].set_xlabel("bruise size (px, log)"); axes[1].set_ylabel("Dice")
axes[1].set_ylim(-0.02,1.02); axes[1].set_title("Dice vs size (all models)", fontsize=10)
axes[1].grid(lw=0.5); axes[1].set_axisbelow(True)

# miss rate by size quartile, per model
q = pd.qcut(sizes, 4, labels=["Q1 smallest","Q2","Q3","Q4 largest"])
size_bin = q.to_frame("size_q")
for n in CORE:
    d = per_image[n].set_index("stem")
    mr = d.join(size_bin).groupby("size_q", observed=True).complete_miss.mean()*100
    axes[2].plot(range(len(mr)), mr.values, "-o", color=PALETTE[n], label=DISP[n], lw=2)
axes[2].set_xticks(range(4)); axes[2].set_xticklabels(["Q1\nsmallest","Q2","Q3","Q4\nlargest"], fontsize=8)
axes[2].set_ylabel("complete-miss rate (%)"); axes[2].set_title("Misses concentrate on small bruises", fontsize=10)
axes[2].grid(lw=0.5); axes[2].set_axisbelow(True); axes[2].legend(fontsize=6.5, frameon=False)
plt.tight_layout(); plt.show()
pd.DataFrame([{"model": DISP[n],
               "spearman_size_vs_dice": spearmanr(per_image[n].gt_positive_pixels, per_image[n].dice)[0],
               "p": spearmanr(per_image[n].gt_positive_pixels, per_image[n].dice)[1]} for n in CORE]).round(4)

## E2 · Recall & precision vs size

In [ ]:
q = pd.qcut(per_image[CORE[0]].set_index("stem").gt_positive_pixels, 5,
            labels=["Q1","Q2","Q3","Q4","Q5"])
qb = q.to_frame("size_q")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
for n in CORE:
    d = per_image[n].set_index("stem").join(qb)
    rec = d.groupby("size_q", observed=True).recall.mean()
    pre = d.groupby("size_q", observed=True).precision.mean()
    a1.plot(range(len(rec)), rec.values, "-o", color=PALETTE[n], lw=2, label=DISP[n])
    a2.plot(range(len(pre)), pre.values, "-o", color=PALETTE[n], lw=2)
for ax, t in [(a1,"Recall vs size quintile"),(a2,"Precision vs size quintile")]:
    ax.set_xticks(range(5)); ax.set_xticklabels(["Q1\nsmall","Q2","Q3","Q4","Q5\nlarge"], fontsize=8)
    ax.set_ylim(0,1.02); ax.set_title(t, fontsize=10.5); ax.grid(lw=0.5); ax.set_axisbelow(True)
a1.set_ylabel("mean recall"); a1.legend(fontsize=6.5, frameon=False, loc="lower right")
plt.tight_layout(); plt.show()
print("Recall falls fastest on the smallest bruises — that is where the misses live.")

## E3 · ⭐ The size↔fairness confound — bruise size by ITA group

In [ ]:
sz = per_image[CORE[0]][["stem","gt_positive_pixels"]].merge(ITA, on="stem")
groups = [g for g in GROUP_ORDER if g in sz.skin_tone_category.unique()]
fig, ax = plt.subplots(figsize=(9.5, 4.6))
data = [sz[sz.skin_tone_category==g].gt_positive_pixels.values for g in groups]
bp = ax.boxplot(data, labels=[g.split(" ")[0] for g in groups], showfliers=False, patch_artist=True)
for patch, c in zip(bp["boxes"], plt.cm.viridis(np.linspace(0,1,len(groups)))):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.set_yscale("log"); ax.set_ylabel("bruise size (px, log)")
ax.set_title("Bruise size by ITA group — is the fairness gap really a size gap?", fontsize=11, pad=10)
ax.grid(axis="y", lw=0.5); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

H, p = kruskal(*data)
tab = sz.groupby("skin_tone_category").gt_positive_pixels.agg(["count","median"]).reindex(groups)
print(f"Kruskal–Wallis (size across ITA groups): H={H:.2f}, p={p:.4f}")
print("If p is small, group-level Dice differences in section D are partly a size artefact,")
print("because smaller bruises are harder for every model (E1). Report size as a covariate.")
tab.round(0)

## E4 · ⭐ Does the skin-tone signal survive size? (stratified + regression)

E3 showed size is confounded with ITA group. This asks the decisive question: is the
light-skin deficit *only* size? Two tests per model — (1) within size terciles, compare
light vs rest on miss-rate and recall (if the gap is pure size, it vanishes once you
hold size roughly fixed); (2) a regression that puts `log10(size)` and a light-skin flag
in together — `recall ~ log_size + light` (OLS, always defined) and
`complete_miss ~ log_size + light` (logistic, where there are enough miss events). **SEs
are clustered by subject** (the light group is only ~9 people; treating 185 images as
independent would make these p-values far too small — the same reason every CI here is a
subject bootstrap). **If the `light` term stays significant with size controlled, the
signal is not only size; if it collapses, the E3 confound explains the section-D gap.**
Still n=28-limited — diagnostic, not proof.

In [ ]:
try:
    import statsmodels.formula.api as smf
    HAVE_SM = True
except Exception:
    HAVE_SM = False
    print("statsmodels unavailable -> regressions skipped (stratified table still runs)")

LIGHT = "Light (II-III)"
strat_rows, coef_rows = [], []
for n in CORE:
    d = per_image[n].merge(ITA, on="stem").merge(SUBJ, on="stem").copy()
    d["light"]    = (d.skin_tone_category == LIGHT).astype(int)
    d["log_size"] = np.log10(d.gt_positive_pixels.clip(lower=1))
    d["miss"]     = d.complete_miss.astype(int)
    d["size_tercile"] = pd.qcut(d.gt_positive_pixels, 3, labels=["small","medium","large"])

    # (1) stratified light-vs-rest within each size tercile (+ overall)
    for terc, sub in list(d.groupby("size_tercile", observed=True)) + [("all", d)]:
        L, R = sub[sub.light == 1], sub[sub.light == 0]
        strat_rows.append({"model": DISP[n], "size_tercile": str(terc),
                           "n_light": len(L), "n_rest": len(R),
                           "miss_light_pct": L.miss.mean()*100 if len(L) else np.nan,
                           "miss_rest_pct":  R.miss.mean()*100 if len(R) else np.nan,
                           "recall_light":   L.recall.mean() if len(L) else np.nan,
                           "recall_rest":    R.recall.mean() if len(R) else np.nan})

    # (2) regression: light term controlling for log size. SEs are CLUSTERED BY SUBJECT --
    # the 185 images are only 28 people, and the light group is ~9 subjects, so an
    # independence assumption would make these p-values far too small (the same reason
    # every CI in this notebook is a subject bootstrap). Clustered SEs are the honest test.
    if HAVE_SM:
        clk = {"cov_type": "cluster", "cov_kwds": {"groups": d["subject"]}}
        try:
            ols = smf.ols("recall ~ log_size + light", data=d).fit(**clk)
            coef_rows.append({"model": DISP[n], "outcome": "recall (OLS, subj-clustered)",
                              "light_coef": ols.params["light"], "light_p": ols.pvalues["light"],
                              "logsize_coef": ols.params["log_size"], "logsize_p": ols.pvalues["log_size"],
                              "light_OR": np.nan})
        except Exception:
            coef_rows.append({"model": DISP[n], "outcome": "recall (OLS): failed", "light_coef": np.nan,
                              "light_p": np.nan, "logsize_coef": np.nan, "logsize_p": np.nan, "light_OR": np.nan})
        # Logistic needs enough miss EVENTS or it quasi-separates and returns garbage
        # (e.g. a model with 1 miss gives OR ~1e10). recall(OLS) carries the signal for
        # the near-zero-miss SegFormer models; the miss logit is only run where it's stable.
        if int(d.miss.sum()) >= 5:
            try:
                import warnings
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    lg = smf.logit("miss ~ log_size + light", data=d).fit(disp=0, **clk)
                coef_rows.append({"model": DISP[n], "outcome": "miss (Logit)",
                                  "light_coef": lg.params["light"], "light_p": lg.pvalues["light"],
                                  "logsize_coef": lg.params["log_size"], "logsize_p": lg.pvalues["log_size"],
                                  "light_OR": float(np.exp(lg.params["light"]))})
            except Exception:
                coef_rows.append({"model": DISP[n], "outcome": "miss (Logit: failed)",
                                  "light_coef": np.nan, "light_p": np.nan, "logsize_coef": np.nan,
                                  "logsize_p": np.nan, "light_OR": np.nan})

SIZE_COND      = pd.DataFrame(strat_rows)
SIZE_COND_COEF = pd.DataFrame(coef_rows)

# figure: miss% by size tercile, light vs rest, for YOLO-direct (where the gap lives)
focus = "yolo_sem_direct"
fd = SIZE_COND[(SIZE_COND.model == DISP[focus]) & (SIZE_COND.size_tercile != "all")]
fig, ax = plt.subplots(figsize=(8, 4.4))
x = np.arange(len(fd)); w = 0.38
ax.bar(x - w/2, fd["miss_light_pct"], w, color="#e34948", label="Light (II-III)")
ax.bar(x + w/2, fd["miss_rest_pct"],  w, color="#2a78d6", label="rest")
for xi, vl, vr in zip(x, fd["miss_light_pct"], fd["miss_rest_pct"]):
    ax.text(xi - w/2, (vl if np.isfinite(vl) else 0)+0.3, f"{vl:.0f}", ha="center", fontsize=8)
    ax.text(xi + w/2, (vr if np.isfinite(vr) else 0)+0.3, f"{vr:.0f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(fd.size_tercile)
ax.set_ylabel("complete-miss rate (%)"); ax.set_xlabel("bruise-size tercile")
ax.set_title(f"{DISP[focus]}: is the light-skin miss gap just size?", fontsize=11, pad=10)
ax.legend(frameon=False); ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print("Stratified light-vs-rest by size tercile (miss% and recall):")
print(SIZE_COND.round(2).to_string(index=False))
print("\nRegression — 'light' term with log10(size) controlled:")
print(SIZE_COND_COEF.round(4).to_string(index=False))
print("\nRead: light_p<0.05 with log_size in the model => skin-tone signal is NOT only size.")
print("      light term collapsing to n.s. => the E3 size confound explains the section-D gap.")
SIZE_COND_COEF.round(4)

---
# F · Annotation ceiling & cost

## F1 · ⭐ The annotation ceiling — do models beat human–human agreement?

Our test masks are a **majority vote of three experts**. How much do the experts even
agree with each other? We read the per-image inter-labeler Dice from
`interlabeler_agreement_640.csv` (pure mask arithmetic, no model). If experts agree at
only ~0.64, a model at ~0.79 vs their consensus is already **past the point where
humans agree with each other** — and a 0.02 Dice gap between two models is far below
the label noise. Note **Paul** (whose masks we *train on*) is the outlier: he matches
the majority least of the three.

In [ ]:
IL = il.copy()
human_pairs = ["paul_vs_gbarimah","paul_vs_erik","gbarimah_vs_erik"]
human_maj   = ["paul_vs_majority","gbarimah_vs_majority","erik_vs_majority"]
rows = []
for c in human_pairs:
    a, b = c.split("_vs_"); rows.append({"label": f"{a.capitalize()} vs {b.capitalize()}", "kind":"human vs human", "dice": IL[c].mean()})
for c in human_maj:
    a = c.split("_vs_")[0]; rows.append({"label": f"{a.capitalize()} vs majority", "kind":"human vs majority", "dice": IL[c].mean()})
for n in CORE:
    rows.append({"label": DISP[n], "kind":"model vs majority", "dice": per_image[n].dice.mean()})
ceiling = pd.DataFrame(rows)
HH = IL[human_pairs].values.mean()

KIND_C = {"human vs human":"#e34948", "human vs majority":"#eb6834", "model vs majority":"#2a78d6"}
fig, ax = plt.subplots(figsize=(9, 5.4))
y = np.arange(len(ceiling))[::-1]
ax.barh(y, ceiling.dice, height=0.62, color=[KIND_C[k] for k in ceiling.kind], zorder=3)
for yy, v in zip(y, ceiling.dice):
    ax.text(v+0.008, yy, f"{v:.3f}", va="center", fontsize=8.5)
ax.axvline(HH, color=MUTED, ls="--", lw=1.5, zorder=2)
ax.text(HH, len(ceiling)-0.2, f"  avg human-human = {HH:.3f}", color=MUTED, fontsize=9, va="top")
ax.set_yticks(y); ax.set_yticklabels(ceiling.label, fontsize=8.5)
ax.set_xlabel("Mean Dice"); ax.set_xlim(0, 1.05)
ax.set_title("Annotation ceiling: every model beats human–human agreement", fontsize=12, pad=10)
ax.grid(axis="x", lw=0.6, zorder=0); ax.set_axisbelow(True)
ax.legend(handles=[Line2D([0],[0],marker="s",lw=0,markerfacecolor=c,markeredgecolor="none",markersize=9,label=k)
                   for k,c in KIND_C.items()], loc="upper center", bbox_to_anchor=(0.5,-0.09), ncol=3, fontsize=8.5, frameon=False)
plt.tight_layout(); plt.show()
ceiling.round(4)

## F2 · Does the model beat the annotator it learned from?

Models train on **Paul's** masks only, scored vs the 3-expert majority. Paul himself
matches that majority at ~0.70. Paired subject-bootstrap Δ (model − Paul, vs the same
majority): if the interval clears zero, the model trained on Paul agrees with consensus
**better than Paul does** — training averages away one annotator's idiosyncrasies.

In [ ]:
base = IL[["stem","paul_vs_majority"]].merge(SUBJ, on="stem")
rows = []
for n in CORE:
    m = base.merge(per_image[n][["stem","dice"]].rename(columns={"dice":"model"}), on="stem")
    r = paired_delta(m, lambda x: x.model.mean() - x.paul_vs_majority.mean(), B=4000)
    rows.append({"model": n, "model_vs_maj": m.model.mean(), "paul_vs_maj": m.paul_vs_majority.mean(),
                 "delta": r["delta"], "ci_lo": r["ci_lo"], "ci_hi": r["ci_hi"], "p_gt0": r["p_gt0"],
                 "beats_paul": r["ci_lo"] > 0})
beat = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8.8, 4.2))
xs = np.arange(len(beat))
ax.bar(xs, beat.delta, width=0.55, color=[PALETTE[n] for n in beat.model], zorder=3)
ax.errorbar(xs, beat.delta, yerr=[beat.delta-beat.ci_lo, beat.ci_hi-beat.delta],
            fmt="none", ecolor=INK, elinewidth=1.6, capsize=5, zorder=4)
ax.axhline(0, color="#c3c2b7", lw=1.2, zorder=2)
for x, d_, hi, p in zip(xs, beat.delta, beat.ci_hi, beat.p_gt0):
    ax.text(x, hi+0.005, f"{d_:+.3f}\nP={p*100:.0f}%", ha="center", fontsize=7.5)
ax.set_xticks(xs); ax.set_xticklabels([DISP[n] for n in beat.model], rotation=18, ha="right", fontsize=7.5)
ax.set_ylabel("Δ mean Dice (model − Paul, vs majority)")
ax.set_title("Every model matches expert consensus better than the annotator it trained on", fontsize=10.5, pad=10)
ax.grid(axis="y", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()
beat.assign(model=lambda d: d.model.map(DISP)).round(4)

## F3 · Speed / cost — accuracy vs latency

From `results_final/benchmark_640.csv` (640-tensor-on-GPU → mask-on-GPU, seed 0, the
architectural number). Pareto view: up-and-left is better. YOLO is far faster but pays
in miss rate; SegFormer-B0 is the balance point.

In [ ]:
try:
    B = BENCH_DF.copy()
    name_map = {"segformer_b2_teacher":"segformer_b2_teacher","segformer_b0_direct":"segformer_b0_direct",
                "segformer_b0_distilled":"segformer_b0_distilled","yolo_sem_direct":"yolo_sem_direct",
                "yolo_sem_distilled":"yolo_sem_distilled"}
    fig, ax = plt.subplots(figsize=(8.6, 5))
    for _, r in B.iterrows():
        n = r["model"]
        if n not in per_image: continue
        med = per_image[n].dice.median()
        ax.scatter(r.fps, med, s=180, color=PALETTE[n], edgecolors=INK, lw=0.7, zorder=3)
        ax.annotate(f"{DISP[n]}\n{r.params_M:.1f}M · p95 {r.p95_ms:.1f}ms",
                    (r.fps, med), fontsize=7.5, textcoords="offset points", xytext=(8,-4))
    ax.set_xlabel("throughput (FPS, higher better)"); ax.set_ylabel("median Dice (higher better)")
    ax.set_title("Accuracy vs speed — best seed Dice × benchmarked FPS", fontsize=11, pad=10)
    ax.grid(lw=0.6); ax.set_axisbelow(True)
    plt.tight_layout(); plt.show()
    print(B[["model","median_ms","p95_ms","fps","params_M"]].to_string(index=False))
except Exception as e:
    print("benchmark_640.csv not read:", e)

## Save every table + figure to Drive

In [ ]:
import datetime, shutil
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ADIR = Path(CFG["drive_dir"]) / f"saved_analysis_{stamp}"
ADIR.mkdir(parents=True, exist_ok=True)
for n in CORE:
    per_image[n].merge(SUBJ, on="stem").merge(ITA, on="stem").to_csv(ADIR/f"per_image_{n}.csv", index=False)
for n in YOLO_MODELS:
    per_image_custom[n].to_csv(ADIR/f"per_image_{n}_custom255.csv", index=False)
head.to_csv(ADIR/"headline.csv", index=False)
FOREST.to_csv(ADIR/"paired_contrasts.csv", index=False)
miss.to_csv(ADIR/"miss_rates.csv", index=False)
beat.to_csv(ADIR/"model_vs_paul.csv", index=False)
ceiling.to_csv(ADIR/"annotation_ceiling.csv", index=False)
FAIR_GROUP.to_csv(ADIR/"fairness_per_group_bestseed.csv", index=False)
FAIR_STATS.to_csv(ADIR/"fairness_stats_bestseed.csv", index=False)
FAIR_PAIR.to_csv(ADIR/"fairness_pairwise_bestseed.csv", index=False)
SIZE_COND.to_csv(ADIR/"size_conditioned_fairness_stratified.csv", index=False)
SIZE_COND_COEF.to_csv(ADIR/"size_conditioned_fairness_regression.csv", index=False)
print("saved ->", ADIR)
for f in sorted(ADIR.glob("*")): print("   ", f.name)

### How to read this analysis

1. **Best-seed, one inference pass** — every figure shares it; 3-seed spread is on disk.
2. **Miss rate is the honest axis** (B1): it separates models by more than label noise.
3. **Fairness (D) is exploratory** at n=28 — direction, not proof; and **size (E3) is a
   confound** you must report as a covariate before claiming a skin-tone gap.
4. **The ceiling (F1) reframes everything**: model-to-model Dice gaps live below the
   ~0.36 of annotation noise, and a model trained on Paul beats Paul vs consensus (F2).